In [ ]:
from google.colab import drive
import os

# 1. Mount your personal Google Drive account to the virtual environment
drive.mount('/content/drive')

# 2. Update your terminal working path to match your project directory location
# (Change 'MyProjectFolder' to match the exact name of your folder inside Drive)
os.chdir('/content/drive/MyDrive/mplad-dvc-storage')

print(f"Current Colab Working Directory verified at: {os.getcwd()}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current Colab Working Directory verified at: /content/drive/MyDrive/mplad-dvc-storage


In [ ]:
# ============================================================
# PHASE 3: Methodological Execution - Detector 1 (Isolation Forest) - FINAL
# ============================================================
import pandas as pd
import numpy as np
import time
import pickle
import os
from sklearn.ensemble import IsolationForest

# 1. Load the preprocessed dataset
print("Loading preprocessed data...")
CSV_PATH = "/content/drive/MyDrive/mplad-dvc-storage/creditcard_preprocessed.csv"
df = pd.read_csv(CSV_PATH)

# Separate features (X) - we do not need y during unsupervised fit
X = df.drop(columns=['Class'])

# 2. Define Parameter Grid (Amended: 0.1 instead of redundant 'auto')
contamination_values = [0.001, 0.005, 0.01, 0.017, 0.05]
n_estimators_values = [100, 200, 500]
max_samples_values = [256, 0.1] # 0.1 = 10% of data (~28,480 samples)

# Storage for results
results_summary = []
raw_predictions = {}

total_combos = len(contamination_values) * len(n_estimators_values) * len(max_samples_values)
print(f"Starting Isolation Forest grid search ({total_combos} combinations)...")

# 3. Iterate through parameter combinations
for contam in contamination_values:
    for n_est in n_estimators_values:
        for max_samp in max_samples_values:
            combo_name = f"if_cont{contam}_nest{n_est}_msamp{max_samp}"
            print(f"  Training: {combo_name} ...", end=" ", flush=True)

            start_time = time.time()

            # Initialize Isolation Forest
            iso_forest = IsolationForest(
                n_estimators=n_est,
                max_samples=max_samp,
                contamination=contam,
                random_state=42, # Fixed for reproducibility
                n_jobs=-1
            )

            # fit_predict returns -1 for anomalies and 1 for inliers
            raw_labels = iso_forest.fit_predict(X)
            # decision_function returns anomaly score (lower = more anomalous)
            scores = iso_forest.decision_function(X)

            exec_time = time.time() - start_time

            # MEMORY OPTIMIZATION:
            # Map labels to 1 (fraud) and 0 (legit), cast to uint8 (1 byte)
            mapped_labels = np.where(raw_labels == -1, 1, 0).astype(np.uint8)
            # Cast scores to float32 (4 bytes) to save memory
            scores = scores.astype(np.float32)

            # Store summary metrics
            results_summary.append({
                'Model': 'IsolationForest',
                'Combo': combo_name,
                'contamination': contam,
                'n_estimators': n_est,
                'max_samples': max_samp,
                'exec_time_seconds': round(exec_time, 2),
                'flagged_anomalies': int(mapped_labels.sum())
            })

            # Store raw predictions for Phase 4 evaluation
            raw_predictions[combo_name] = {
                'labels': mapped_labels,
                'scores': scores
            }

            # Explicitly delete large objects to help garbage collector
            del iso_forest, raw_labels

            print(f"Done in {exec_time:.2f}s (Flagged {int(mapped_labels.sum())} anomalies)")

# 4. Save outputs directly to your permanent Google Drive folder
print("\nSaving execution summary and raw predictions...")

# Define the central directory path to your Google Drive storage folder
STORAGE_DIR = "/content/drive/MyDrive/mplad-dvc-storage"

# Save summary to CSV in Google Drive
summary_df = pd.DataFrame(results_summary)
output_csv_path = os.path.join(STORAGE_DIR, "iforest_execution_summary.csv")
summary_df.to_csv(output_csv_path, index=False)

# Save raw predictions to a pickle file in Google Drive for Phase 4
output_pkl_path = os.path.join(STORAGE_DIR, "iforest_raw_predictions.pkl")
with open(output_pkl_path, "wb") as f:
    pickle.dump(raw_predictions, f, protocol=pickle.HIGHEST_PROTOCOL)

print("\n=== ISOLATION FOREST EXECUTION SUMMARY ===")
print(summary_df.to_string(index=False))
print(f"\nStep 1 Complete. Files safely stored in: {STORAGE_DIR}")


Loading preprocessed data...
Starting Isolation Forest grid search (30 combinations)...
  Training: if_cont0.001_nest100_msamp256 ... Done in 6.61s (Flagged 285 anomalies)
  Training: if_cont0.001_nest100_msamp0.1 ... Done in 11.08s (Flagged 285 anomalies)
  Training: if_cont0.001_nest200_msamp256 ... Done in 11.01s (Flagged 285 anomalies)
  Training: if_cont0.001_nest200_msamp0.1 ... Done in 21.85s (Flagged 285 anomalies)
  Training: if_cont0.001_nest500_msamp256 ... Done in 28.85s (Flagged 285 anomalies)
  Training: if_cont0.001_nest500_msamp0.1 ... Done in 55.49s (Flagged 285 anomalies)
  Training: if_cont0.005_nest100_msamp256 ... Done in 5.21s (Flagged 1424 anomalies)
  Training: if_cont0.005_nest100_msamp0.1 ... Done in 11.22s (Flagged 1425 anomalies)
  Training: if_cont0.005_nest200_msamp256 ... Done in 11.79s (Flagged 1425 anomalies)
  Training: if_cont0.005_nest200_msamp0.1 ... Done in 20.37s (Flagged 1425 anomalies)
  Training: if_cont0.005_nest500_msamp256 ... Done in 29.07s

In [ ]:
# ============================================================
# PHASE 3: Methodological Execution - Detector 2 (LOF) - COLAB FIXED
# ============================================================
import pandas as pd
import numpy as np
import time
import pickle
import os
from sklearn.neighbors import LocalOutlierFactor

# Define the central directory path to your Google Drive storage folder
STORAGE_DIR = "/content/drive/MyDrive/mplad-dvc-storage"

# 1. Load the preprocessed dataset
print("Loading preprocessed data...")
CSV_PATH = os.path.join(STORAGE_DIR, "creditcard_preprocessed.csv")
df = pd.read_csv(CSV_PATH)

# ============================================================
# STRATIFIED SUBSAMPLING TO PREVENT INTRACTABLE COMPUTATION
# Complexity drops from 81 Billion calculations to under 1.6 Billion!
# ============================================================
print("Applying stratified sub-sampling to keep training feasible on Colab...")
fraud_data = df[df['Class'] == 1]
legit_data = df[df['Class'] == 0]

# Sample 40,000 normal transactions and combine with all available fraud cases
sampled_legit = legit_data.sample(n=40000, random_state=42)
df_sampled = pd.concat([sampled_legit, fraud_data], ignore_index=True)

# Separate features (X) - unsupervised training matrix
X = df_sampled.drop(columns=['Class'])
total_rows = len(X)
print(f"Original Row Count: {len(df)} | Sub-sampled Training Rows: {total_rows}")

# 2. Define Parameter Grid
n_neighbors_values = [5, 10, 20, 50]
contamination_values = [0.001, 0.005, 0.01, 0.017, 0.05]
metrics = ['euclidean', 'manhattan']

# Storage for results
results_summary = []
raw_predictions = {}

total_combos = len(n_neighbors_values) * len(contamination_values) * len(metrics)
print(f"Starting LOF grid search ({total_combos} combinations). Running safely now...")

# 3. Iterate through parameter combinations
for n_neigh in n_neighbors_values:
    for contam in contamination_values:
        for met in metrics:
            combo_name = f"lof_n{n_neigh}_cont{contam}_met{met}"
            print(f"  Training: {combo_name} ...", end=" ", flush=True)

            start_time = time.time()

            # Initialize LOF for Outlier Detection
            # FIX: n_jobs=1 or 2 stops the Colab/Joblib threading deadlock bug
            lof = LocalOutlierFactor(
                n_neighbors=n_neigh,
                contamination=contam,
                metric=met,
                algorithm='ball_tree',
                leaf_size=40,
                n_jobs=2
            )

            # fit_predict runs the accelerated tree density checks
            raw_labels = lof.fit_predict(X)
            scores = lof.negative_outlier_factor_

            exec_time = time.time() - start_time

            # MEMORY OPTIMIZATION:
            mapped_labels = np.where(raw_labels == -1, 1, 0).astype(np.uint8)
            scores = scores.astype(np.float32)

            # Store summary metrics
            results_summary.append({
                'Model': 'LOF',
                'Combo': combo_name,
                'n_neighbors': n_neigh,
                'contamination': contam,
                'metric': met,
                'exec_time_seconds': round(exec_time, 2),
                'flagged_anomalies': int(mapped_labels.sum()),
                'flagged_anomalies_pct': round(mapped_labels.sum() / total_rows * 100, 4)
            })

            # Store raw predictions for Phase 4 evaluation
            raw_predictions[combo_name] = {
                'labels': mapped_labels,
                'scores': scores
            }

            # Explicitly delete large objects to help garbage collector
            del lof, raw_labels

            print(f"Done in {exec_time:.2f}s (Flagged {int(mapped_labels.sum())} anomalies)")

# 4. Save outputs directly to your permanent Google Drive folder
print("\nSaving execution summary and raw predictions...")

# Save summary to CSV in Google Drive
summary_df = pd.DataFrame(results_summary)
output_csv_path = os.path.join(STORAGE_DIR, "lof_execution_summary.csv")
summary_df.to_csv(output_csv_path, index=False)

# Save raw predictions to a pickle file in Google Drive for Phase 4
output_pkl_path = os.path.join(STORAGE_DIR, "lof_raw_predictions.pkl")
with open(output_pkl_path, "wb") as f:
    pickle.dump(raw_predictions, f, protocol=pickle.HIGHEST_PROTOCOL)

print("\n=== LOF EXECUTION SUMMARY ===")
print(summary_df.to_string(index=False))
print(f"\nStep 2 Complete. Files safely stored in: {STORAGE_DIR}")


Loading preprocessed data...
Applying stratified sub-sampling to keep training feasible on Colab...
Original Row Count: 284807 | Sub-sampled Training Rows: 40492
Starting LOF grid search (40 combinations). Running safely now...
  Training: lof_n5_cont0.001_meteuclidean ... Done in 536.44s (Flagged 41 anomalies)
  Training: lof_n5_cont0.001_metmanhattan ... Done in 561.14s (Flagged 41 anomalies)
  Training: lof_n5_cont0.005_meteuclidean ... Done in 517.45s (Flagged 203 anomalies)
  Training: lof_n5_cont0.005_metmanhattan ... Done in 547.04s (Flagged 203 anomalies)
  Training: lof_n5_cont0.01_meteuclidean ... Done in 515.82s (Flagged 405 anomalies)
  Training: lof_n5_cont0.01_metmanhattan ... Done in 571.86s (Flagged 405 anomalies)
  Training: lof_n5_cont0.017_meteuclidean ... Done in 520.64s (Flagged 689 anomalies)
  Training: lof_n5_cont0.017_metmanhattan ... Done in 553.86s (Flagged 689 anomalies)
  Training: lof_n5_cont0.05_meteuclidean ... Done in 502.43s (Flagged 2025 anomalies)
  

In [ ]:
# ============================================================
# PHASE 3: Methodological Execution
# ============================================================
import os
import pandas as pd
import numpy as np
import time
import pickle
import gc
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

# Define the central directory path to your Google Drive storage folder
STORAGE_DIR = "/content/drive/MyDrive/mplad-dvc-storage"
FS_PATH = os.path.join(STORAGE_DIR, "ULB_FS_A_Baseline.csv")

# 1. Load Frozen Phase 2 Feature Space from Google Drive
print(f"Loading frozen feature space from {FS_PATH} ...")
fs_df = pd.read_csv(FS_PATH)

# Extract row_id once globally
row_ids = fs_df['row_id'].to_numpy()
X = fs_df.drop(columns=['row_id']).to_numpy() # Convert to numpy for strict memory control

# 2. Define Decision Threshold (Top 1% most anomalous)
THRESHOLD_PERCENTILE = 1.0

run_metadata = []
# Storing raw numpy arrays in a flat dict to eliminate DataFrame overhead
canonical_predictions = {}

# ==========================================
# DETECTOR 1: Isolation Forest
# ==========================================
print("\n--- STARTING ISOLATION FOREST ---")
# Amended: max_samples changed from 'auto' to 0.05. n_jobs=2 for thread safety.
if_configs = [
    {'n_estimators': 100, 'max_samples': 256, 'random_state': 42},
    {'n_estimators': 200, 'max_samples': 256, 'random_state': 42}, # Param sensitivity
    {'n_estimators': 100, 'max_samples': 0.05, 'random_state': 42}, # Param sensitivity (5% sub-sample)
    {'n_estimators': 100, 'max_samples': 256, 'random_state': 43}, # Seed stability
    {'n_estimators': 100, 'max_samples': 256, 'random_state': 44}  # Seed stability
]

for i, cfg in enumerate(if_configs):
    run_id = f"IF_{i+1:03d}"
    print(f"  Running {run_id} (n_est={cfg['n_estimators']}, msamp={cfg['max_samples']}, seed={cfg['random_state']}) ...", end=" ", flush=True)

    start = time.time()
    iso = IsolationForest(n_estimators=cfg['n_estimators'], max_samples=cfg['max_samples'],
                          random_state=cfg['random_state'], n_jobs=2)

    iso.fit(X)
    raw_scores = iso.decision_function(X)
    exec_t = time.time() - start

    threshold_val = np.percentile(raw_scores, THRESHOLD_PERCENTILE)
    flags = np.where(raw_scores <= threshold_val, 1, 0).astype(np.uint8)
    scores_f32 = raw_scores.astype(np.float32)

    flagged_count = int(flags.sum())

    run_metadata.append({
        'run_id': run_id, 'model': 'IsolationForest',
        'n_estimators': cfg['n_estimators'], 'max_samples': cfg['max_samples'],
        'random_state': cfg['random_state'], 'runtime_seconds': round(exec_t, 2),
        'threshold_rule': f'Bottom {THRESHOLD_PERCENTILE}% of scores',
        'flagged_count': flagged_count
    })

    # Store lightweight numpy arrays
    canonical_predictions[run_id] = {
        'scores': scores_f32,
        'flags': flags
    }

    print(f"Done in {exec_t:.2f}s (Flagged: {flagged_count})")
    # Memory cleanup at absolute boundary
    del iso, raw_scores, flags, scores_f32, flagged_count
    gc.collect()

# ==========================================
# DETECTOR 2: Local Outlier Factor
# ==========================================
print("\n--- STARTING LOCAL OUTLIER FACTOR ---")
# Amended: n_jobs=2 to prevent Colab/backend deadlocks
lof_configs = [
    {'n_neighbors': 5, 'metric': 'euclidean'},
    {'n_neighbors': 15, 'metric': 'euclidean'},
    {'n_neighbors': 30, 'metric': 'euclidean'},
    {'n_neighbors': 50, 'metric': 'euclidean'},
    {'n_neighbors': 30, 'metric': 'manhattan'}
]

for i, cfg in enumerate(lof_configs):
    run_id = f"LOF_{i+1:03d}"
    print(f"  Running {run_id} (n_neigh={cfg['n_neighbors']}, met={cfg['metric']}) ...", end=" ", flush=True)

    start = time.time()
    lof = LocalOutlierFactor(n_neighbors=cfg['n_neighbors'], metric=cfg['metric'],
                             algorithm='ball_tree', leaf_size=40, n_jobs=2)

    lof.fit_predict(X)
    raw_scores = lof.negative_outlier_factor_
    exec_t = time.time() - start

    threshold_val = np.percentile(raw_scores, THRESHOLD_PERCENTILE)
    flags = np.where(raw_scores <= threshold_val, 1, 0).astype(np.uint8)
    scores_f32 = raw_scores.astype(np.float32)

    flagged_count = int(flags.sum())

    run_metadata.append({
        'run_id': run_id, 'model': 'LOF',
        'n_neighbors': cfg['n_neighbors'], 'metric': cfg['metric'],
        'random_state': 'N/A', 'runtime_seconds': round(exec_t, 2),
        'threshold_rule': f'Bottom {THRESHOLD_PERCENTILE}% of scores',
        'flagged_count': flagged_count
    })

    # Store lightweight numpy arrays
    canonical_predictions[run_id] = {
        'scores': scores_f32,
        'flags': flags
    }

    print(f"Done in {exec_t:.2f}s (Flagged: {flagged_count})")
    # Memory cleanup at absolute boundary
    del lof, raw_scores, flags, scores_f32, flagged_count
    gc.collect()

# ==========================================
# SAVE ARTIFACTS DIRECTLY TO GOOGLE DRIVE
# ==========================================
print("\nSaving metadata and canonical predictions to permanent storage...")

metadata_df = pd.DataFrame(run_metadata)
OUTPUT_CSV_PATH = os.path.join(STORAGE_DIR, "phase3_run_metadata.csv")
metadata_df.to_csv(OUTPUT_CSV_PATH, index=False)

# Bundle row_ids with the predictions dict so they are inseparable
output_bundle = {
    'row_ids': row_ids,
    'predictions': canonical_predictions
}
OUTPUT_PKL_PATH = os.path.join(STORAGE_DIR, "phase3_canonical_predictions.pkl")
with open(OUTPUT_PKL_PATH, "wb") as f:
    pickle.dump(output_bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

print("\n=== PHASE 3 RUN METADATA ===")
print(metadata_df.to_string(index=False))
print(f"\nPhase 3 Complete. Strict unsupervised outputs safely stored in: {STORAGE_DIR}")


Loading ground truth...
Loading & evaluating iForest predictions...
Loading & evaluating LOF predictions...
Metrics table successfully saved to permanent storage: /content/drive/MyDrive/mplad-dvc-storage/phase3_evaluation_metrics.csv

=== BEST ISOLATION FOREST CONFIGURATION ===
Combo        if_cont0.005_nest100_msamp0.1
Precision                           0.1986
Recall                              0.5752
F1                                  0.2953
ROC_AUC                             0.9514
PR_AUC                               0.234
TP                                     283
FP                                    1142
Name: 7, dtype: object

=== BEST LOF CONFIGURATION ===
Combo        lof_n5_cont0.01_meteuclidean
Precision                          0.0074
Recall                             0.0286
F1                                 0.0118
ROC_AUC                            0.5401
PR_AUC                             0.0033
TP                                      3
FP                          

In [ ]:
# ============================================================
# PHASE 3: Methodological Execution
# ============================================================
import os
import pandas as pd
import numpy as np
import time
import pickle
import gc
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

# Define the central directory path to your Google Drive storage folder
STORAGE_DIR = "/content/drive/MyDrive/mplad-dvc-storage"
FS_PATH = os.path.join(STORAGE_DIR, "ULB_FS_A_Baseline.csv")

# 1. Load Frozen Phase 2 Feature Space from Google Drive
print(f"Loading frozen feature space from {FS_PATH}...")
fs_df = pd.read_csv(FS_PATH)

# 2. Unsupervised Subsample (30,000 rows) for Colab compatibility
# This maintains the same baseline X for both IF and LOF
np.random.seed(42)
sample_indices = np.random.choice(len(fs_df), size=30000, replace=False)
fs_subsampled = fs_df.iloc[sample_indices].reset_index(drop=True)

# Extract row_id and X for the 30k sample
row_ids = fs_subsampled['row_id'].to_numpy()
X = fs_subsampled.drop(columns=['row_id']).to_numpy()

THRESHOLD_PERCENTILE = 1.0
run_metadata = []
canonical_predictions = {}

# ==========================================
# DETECTOR 1: Isolation Forest
# ==========================================
print("\n--- STARTING ISOLATION FOREST (30k sample) ---")
if_configs = [
    {'n_estimators': 100, 'max_samples': 256, 'random_state': 42},
    {'n_estimators': 200, 'max_samples': 256, 'random_state': 42},
    {'n_estimators': 100, 'max_samples': 0.05, 'random_state': 42},
    {'n_estimators': 100, 'max_samples': 256, 'random_state': 43},
    {'n_estimators': 100, 'max_samples': 256, 'random_state': 44}
]

for i, cfg in enumerate(if_configs):
    run_id = f"IF_{i+1:03d}"
    print(f"  Running {run_id} ...", end=" ", flush=True)

    start = time.time()
    # n_jobs=1 to prevent any Colab deadlocks
    iso = IsolationForest(n_estimators=cfg['n_estimators'], max_samples=cfg['max_samples'],
                          random_state=cfg['random_state'], n_jobs=1)
    iso.fit(X)
    raw_scores = iso.decision_function(X)
    exec_t = time.time() - start

    threshold_val = np.percentile(raw_scores, THRESHOLD_PERCENTILE)
    flags = np.where(raw_scores <= threshold_val, 1, 0).astype(np.uint8)
    scores_f32 = raw_scores.astype(np.float32)

    run_metadata.append({
        'run_id': run_id, 'model': 'IsolationForest',
        'n_estimators': cfg['n_estimators'], 'max_samples': cfg['max_samples'],
        'random_state': cfg['random_state'], 'runtime_seconds': round(exec_t, 2),
        'threshold_rule': f'Bottom {THRESHOLD_PERCENTILE}% of scores',
        'flagged_count': int(flags.sum())
    })
    canonical_predictions[run_id] = {'scores': scores_f32, 'flags': flags}

    print(f"Done in {exec_t:.2f}s")
    del iso, raw_scores, flags, scores_f32
    gc.collect()

# ==========================================
# DETECTOR 2: Local Outlier Factor
# ==========================================
print("\n--- STARTING LOCAL OUTLIER FACTOR (30k sample) ---")
# Using standard fit_predict since we are on a manageable 30k rows now
lof_configs = [
    {'n_neighbors': 5, 'metric': 'euclidean'},
    {'n_neighbors': 15, 'metric': 'euclidean'},
    {'n_neighbors': 30, 'metric': 'euclidean'},
    {'n_neighbors': 50, 'metric': 'euclidean'},
    {'n_neighbors': 30, 'metric': 'manhattan'}
]

for i, cfg in enumerate(lof_configs):
    run_id = f"LOF_{i+1:03d}"
    print(f"  Running {run_id} (n_neigh={cfg['n_neighbors']}, met={cfg['metric']}) ...", end=" ", flush=True)

    start = time.time()
    # n_jobs=1 to prevent any Colab deadlocks
    lof = LocalOutlierFactor(n_neighbors=cfg['n_neighbors'], metric=cfg['metric'],
                             algorithm='ball_tree', leaf_size=40, n_jobs=1)

    lof.fit_predict(X)
    raw_scores = lof.negative_outlier_factor_
    exec_t = time.time() - start

    threshold_val = np.percentile(raw_scores, THRESHOLD_PERCENTILE)
    flags = np.where(raw_scores <= threshold_val, 1, 0).astype(np.uint8)
    scores_f32 = raw_scores.astype(np.float32)

    run_metadata.append({
        'run_id': run_id, 'model': 'LOF',
        'n_neighbors': cfg['n_neighbors'], 'metric': cfg['metric'],
        'random_state': 'N/A', 'runtime_seconds': round(exec_t, 2),
        'threshold_rule': f'Bottom {THRESHOLD_PERCENTILE}% of scores',
        'flagged_count': int(flags.sum())
    })
    canonical_predictions[run_id] = {'scores': scores_f32, 'flags': flags}

    print(f"Done in {exec_t:.2f}s")
    del lof, raw_scores, flags, scores_f32
    gc.collect()

# ==========================================
# SAVE ARTIFACTS DIRECTLY TO GOOGLE DRIVE
# ==========================================
print("\nSaving metadata and canonical predictions to permanent storage...")
metadata_df = pd.DataFrame(run_metadata)

OUTPUT_CSV_PATH = os.path.join(STORAGE_DIR, "phase3_run_metadata.csv")
metadata_df.to_csv(OUTPUT_CSV_PATH, index=False)

output_bundle = {'row_ids': row_ids, 'predictions': canonical_predictions}
OUTPUT_PKL_PATH = os.path.join(STORAGE_DIR, "phase3_canonical_predictions.pkl")
with open(OUTPUT_PKL_PATH, "wb") as f:
    pickle.dump(output_bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

print("\n=== PHASE 3 RUN METADATA ===")
print(metadata_df.to_string(index=False))
print(f"\nPhase 3 Complete. Strict unsupervised outputs safely stored in: {STORAGE_DIR}")


Loading frozen feature space from /content/drive/MyDrive/mplad-dvc-storage/ULB_FS_A_Baseline.csv...

--- STARTING ISOLATION FOREST (30k sample) ---
  Running IF_001 ... Done in 2.18s
  Running IF_002 ... Done in 5.55s
  Running IF_003 ... Done in 3.58s
  Running IF_004 ... Done in 2.15s
  Running IF_005 ... Done in 2.30s

--- STARTING LOCAL OUTLIER FACTOR (30k sample) ---
  Running LOF_001 (n_neigh=5, met=euclidean) ... Done in 409.05s
  Running LOF_002 (n_neigh=15, met=euclidean) ... Done in 126.68s
  Running LOF_003 (n_neigh=30, met=euclidean) ... Done in 118.37s
  Running LOF_004 (n_neigh=50, met=euclidean) ... Done in 57.04s
  Running LOF_005 (n_neigh=30, met=manhattan) ... Done in 60.82s

Saving metadata and canonical predictions to permanent storage...

=== PHASE 3 RUN METADATA ===
 run_id           model  n_estimators  max_samples random_state  runtime_seconds        threshold_rule  flagged_count  n_neighbors    metric
 IF_001 IsolationForest         100.0       256.00          

In [ ]:
# ============================================================
# PHASE 3 AUDIT & PHASE 4: Label-Based Validation
# ============================================================
import os
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix

# 1. Define Paths pointed directly to your Google Drive storage folder
STORAGE_DIR = "/content/drive/MyDrive/mplad-dvc-storage"
PRED_PATH = os.path.join(STORAGE_DIR, "phase3_canonical_predictions.pkl")
TRUTH_PATH = os.path.join(STORAGE_DIR, "ground_truth_registry.csv")
METADATA_PATH = os.path.join(STORAGE_DIR, "phase3_run_metadata.csv")

# 2. Load Data
print("Loading Phase 3 predictions and ground truth...")
with open(PRED_PATH, "rb") as f:
    output_bundle = pickle.load(f)

row_ids = output_bundle['row_ids']
predictions = output_bundle['predictions']

ground_truth = pd.read_csv(TRUTH_PATH)
metadata_df = pd.read_csv(METADATA_PATH)

# ==========================================
# PHASE 3 MANDATORY AUDIT (Per ChatGPT)
# ==========================================
print("\n--- PHASE 3 AUDIT ---")
assert len(row_ids) == 30000, "Row count is not 30,000!"
assert len(np.unique(row_ids)) == 30000, "row_ids are not unique!"
assert ground_truth['row_id'].is_unique, "Ground truth has duplicate row_ids!"

eval_truth = ground_truth[ground_truth['row_id'].isin(row_ids)].copy()
assert len(eval_truth) == 30000, "Not all 30k row_ids matched in ground truth!"

# Calculate actual fraud prevalence in the 30k sample
sample_fraud_count = eval_truth['Class'].sum()
sample_prevalence = sample_fraud_count / 30000
print(f"30k Sample Fraud Prevalence: {sample_fraud_count} frauds ({sample_prevalence:.4%})")

for run_id, data in predictions.items():
    assert len(data['flags']) == 30000, f"{run_id} does not have 30,000 rows!"
    assert data['flags'].sum() == 300, f"{run_id} did not flag exactly 300 observations!"
print("All Phase 3 assertions passed. Sample is fully aligned and correct.")

# ==========================================
# PHASE 4: TWO-LAYER EVALUATION
# ==========================================
print("\n--- PHASE 4 EVALUATION ---")
evaluation_results = []

# Merge ground truth to the evaluation sample once
eval_truth = eval_truth.sort_values('row_id').reset_index(drop=True)
y_true = eval_truth['Class'].to_numpy()

for run_id, data in predictions.items():
    # 1. Canonical Score Inversion (Higher = More Anomalous)
    # IF decision_function: lower = anomalous. LOF negative_outlier_factor_: lower = anomalous.
    # We invert both so HIGHER = More Anomalous.
    raw_scores = data['scores']
    canonical_scores = -raw_scores

    flags = data['flags']

    # Layer 1: Ranking Quality (Continuous Scores)
    ap = average_precision_score(y_true, canonical_scores)
    roc = roc_auc_score(y_true, canonical_scores)

    # Layer 2: 1% Operating Point (Binary Flags)
    prec = precision_score(y_true, flags, zero_division=0)
    rec = recall_score(y_true, flags, zero_division=0)
    f1 = f1_score(y_true, flags, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_true, flags, labels=[0, 1]).ravel()

    # Fetch metadata
    meta = metadata_df[metadata_df['run_id'] == run_id].iloc[0]

    evaluation_results.append({
        'run_id': run_id,
        'model': meta['model'],
        'config': f"n_est={meta.get('n_estimators', 'N/A')}, msamp={meta.get('max_samples', 'N/A')}, n_neigh={meta.get('n_neighbors', 'N/A')}, met={meta.get('metric', 'N/A')}",
        'seed': meta['random_state'],
        'AP (Layer 1)': round(ap, 4),
        'ROC_AUC (Layer 1)': round(roc, 4),
        'Precision @1%': round(prec, 4),
        'Recall @1%': round(rec, 4),
        'F1 @1%': round(f1, 4),
        'TP': tp, 'FP': fp, 'FN': fn
    })

# 3. Display Final Comparison Table
eval_df = pd.DataFrame(evaluation_results)
OUTPUT_METRICS_PATH = os.path.join(STORAGE_DIR, "phase4_validation_metrics.csv")
eval_df.to_csv(OUTPUT_METRICS_PATH, index=False)

print("\n=== PHASE 4 VALIDATION METRICS ===")
print(eval_df.to_string(index=False))


Loading Phase 3 predictions and ground truth...

--- PHASE 3 AUDIT ---
30k Sample Fraud Prevalence: 47 frauds (0.1567%)
All Phase 3 assertions passed. Sample is fully aligned and correct.

--- PHASE 4 EVALUATION ---

=== PHASE 4 VALIDATION METRICS ===
 run_id           model                                            config  seed  AP (Layer 1)  ROC_AUC (Layer 1)  Precision @1%  Recall @1%  F1 @1%  TP  FP  FN
 IF_001 IsolationForest    n_est=100.0, msamp=256.0, n_neigh=nan, met=nan  42.0        0.0016             0.5061         0.0000      0.0000  0.0000   0 300  47
 IF_002 IsolationForest    n_est=200.0, msamp=256.0, n_neigh=nan, met=nan  42.0        0.0017             0.5121         0.0000      0.0000  0.0000   0 300  47
 IF_003 IsolationForest     n_est=100.0, msamp=0.05, n_neigh=nan, met=nan  42.0        0.0016             0.5102         0.0000      0.0000  0.0000   0 300  47
 IF_004 IsolationForest    n_est=100.0, msamp=256.0, n_neigh=nan, met=nan  43.0        0.0017             0.

In [ ]:
# ============================================================
# PHASE 4 CORRECTED: Aligned Label-Based Validation
# ============================================================
import os
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix

# 1. Define Paths for Google Colab Drive
STORAGE_DIR = "/content/drive/MyDrive/mplad-dvc-storage"
PRED_PATH = os.path.join(STORAGE_DIR, "phase3_canonical_predictions.pkl")
TRUTH_PATH = os.path.join(STORAGE_DIR, "ground_truth_registry.csv")
METADATA_PATH = os.path.join(STORAGE_DIR, "phase3_run_metadata.csv")
OUTPUT_PATH = os.path.join(STORAGE_DIR, "phase4_validation_metrics_CORRECTED.csv")

# 2. Load Data
print("Loading Phase 3 predictions and ground truth...")
with open(PRED_PATH, "rb") as f:
    output_bundle = pickle.load(f)

row_ids = output_bundle['row_ids']
predictions = output_bundle['predictions']

ground_truth = pd.read_csv(TRUTH_PATH)
metadata_df = pd.read_csv(METADATA_PATH)

# ==========================================
# PHASE 4: ALIGNED EVALUATION
# ==========================================
print("\n--- PHASE 4 ALIGNED EVALUATION ---")

# 1. Create the Master DataFrame to guarantee alignment
eval_df = pd.DataFrame({'row_id': row_ids})

# Merge ground truth using row_id to ensure perfect 1-to-1 matching
eval_df = eval_df.merge(ground_truth[['row_id', 'Class']], on='row_id', how='left')

# Verify merge
assert len(eval_df) == 30000, "Row count mismatch!"
assert eval_df['Class'].notna().all(), "Missing ground truth labels!"

y_true = eval_df['Class'].to_numpy()
sample_fraud_count = int(y_true.sum())
print(f"30k Sample Fraud Prevalence: {sample_fraud_count} frauds ({sample_fraud_count/30000:.4%})")

evaluation_results = []

for run_id, data in predictions.items():
    # Extract arrays in their original subsample order
    raw_scores = data['scores']
    flags = data['flags']

    # Verify flags
    assert flags.sum() == 300, f"{run_id} did not flag exactly 300 observations!"

    # Canonical Score Inversion (Higher = More Anomalous)
    canonical_scores = -raw_scores

    # Layer 1: Ranking Quality
    ap = average_precision_score(y_true, canonical_scores)
    roc = roc_auc_score(y_true, canonical_scores)

    # Layer 2: 1% Operating Point
    prec = precision_score(y_true, flags, zero_division=0)
    rec = recall_score(y_true, flags, zero_division=0)
    f1 = f1_score(y_true, flags, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_true, flags, labels=[0, 1]).ravel()

    meta = metadata_df[metadata_df['run_id'] == run_id].iloc[0]

    evaluation_results.append({
        'run_id': run_id,
        'model': meta['model'],
        'config': f"n_est={meta.get('n_estimators', 'N/A')}, msamp={meta.get('max_samples', 'N/A')}, n_neigh={meta.get('n_neighbors', 'N/A')}, met={meta.get('metric', 'N/A')}",
        'seed': meta['random_state'],
        'AP': round(ap, 4),
        'ROC_AUC': round(roc, 4),
        'Prec @1%': round(prec, 4),
        'Rec @1%': round(rec, 4),
        'F1 @1%': round(f1, 4),
        'TP': tp, 'FP': fp, 'FN': fn
    })

# 3. Display and Save Final Table
results_df = pd.DataFrame(evaluation_results)
results_df.to_csv(OUTPUT_PATH, index=False)

print(f"\nCorrected metrics saved to: {OUTPUT_PATH}")
print("\n=== PHASE 4 VALIDATION METRICS (CORRECTED) ===")
print(results_df.to_string(index=False))

Loading Phase 3 predictions and ground truth...

--- PHASE 4 ALIGNED EVALUATION ---
30k Sample Fraud Prevalence: 47 frauds (0.1567%)

Corrected metrics saved to: /content/drive/MyDrive/mplad-dvc-storage/phase4_validation_metrics_CORRECTED.csv

=== PHASE 4 VALIDATION METRICS (CORRECTED) ===
 run_id           model                                            config  seed     AP  ROC_AUC  Prec @1%  Rec @1%  F1 @1%  TP  FP  FN
 IF_001 IsolationForest    n_est=100.0, msamp=256.0, n_neigh=nan, met=nan  42.0 0.2063   0.9778    0.0967   0.6170  0.1671  29 271  18
 IF_002 IsolationForest    n_est=200.0, msamp=256.0, n_neigh=nan, met=nan  42.0 0.1556   0.9797    0.0967   0.6170  0.1671  29 271  18
 IF_003 IsolationForest     n_est=100.0, msamp=0.05, n_neigh=nan, met=nan  42.0 0.3126   0.9803    0.1000   0.6383  0.1729  30 270  17
 IF_004 IsolationForest    n_est=100.0, msamp=256.0, n_neigh=nan, met=nan  43.0 0.2094   0.9787    0.1067   0.6809  0.1844  32 268  15
 IF_005 IsolationForest    n_est=1

In [ ]:
# ============================================================
# PHASE 6: Cross-Method Consistency & Agreement Analysis
# ============================================================
import os
import pandas as pd
import numpy as np
import pickle
from scipy.stats import spearmanr
from scipy.stats import hypergeom

# 1. Define Paths & Load Data
STORAGE_DIR = "/content/drive/MyDrive/mplad-dvc-storage"
PRED_PATH = os.path.join(STORAGE_DIR, "phase3_canonical_predictions.pkl")
TRUTH_PATH = os.path.join(STORAGE_DIR, "ground_truth_registry.csv")

print("Loading data for Phase 6...")
with open(PRED_PATH, "rb") as f:
    output_bundle = pickle.load(f)

row_ids = output_bundle['row_ids']
predictions = output_bundle['predictions']
ground_truth = pd.read_csv(TRUTH_PATH)

# 2. Create Master DataFrame with Canonical Scores (Higher = More Anomalous)
df = pd.DataFrame({'row_id': row_ids})
df = df.merge(ground_truth[['row_id', 'Class']], on='row_id', how='left')

# Extract and invert scores (Higher = More Anomalous)
for run_id, data in predictions.items():
    df[f"{run_id}_score"] = -data['scores']
    df[f"{run_id}_flag"] = data['flags']

# Define runs of interest
if_runs = ['IF_001', 'IF_003', 'IF_004', 'IF_005']
lof_run = 'LOF_004'

# ==========================================
# PART 1: IF Internal Seed Stability
# ==========================================
print("\n=== 1. IF SEED STABILITY (Spearman Rank Correlation) ===")
seed_pairs = [('IF_001', 'IF_004'), ('IF_001', 'IF_005'), ('IF_004', 'IF_005')]
for r1, r2 in seed_pairs:
    corr, _ = spearmanr(df[f"{r1}_score"], df[f"{r2}_score"])
    print(f"{r1} vs {r2}: {corr:.4f}")

print("\n=== 2. IF SEED STABILITY (Top-K Jaccard) ===")
k_values = [30, 150, 300, 600, 1500] # 0.1%, 0.5%, 1%, 2%, 5%
def top_k_jaccard(s1, s2, k):
    top1 = set(np.argsort(s1)[-k:])
    top2 = set(np.argsort(s2)[-k:])
    inter = len(top1 & top2)
    union = len(top1 | top2)
    return inter / union if union > 0 else 0

print(f"{'Pair':<20} | " + " | ".join([f"k={k}" for k in k_values]))
print("-" * 70)
for r1, r2 in seed_pairs:
    jacs = [f"{top_k_jaccard(df[f'{r1}_score'], df[f'{r2}_score'], k):.4f}" for k in k_values]
    print(f"{r1} vs {r2:<12} | " + " | ".join(jacs))

# ==========================================
# PART 2: IF vs LOF Agreement
# ==========================================
print("\n=== 3. IF vs LOF (Spearman Rank Correlation) ===")
for if_run in if_runs:
    corr, _ = spearmanr(df[f"{if_run}_score"], df[f"{lof_run}_score"])
    print(f"{if_run} vs {lof_run}: {corr:.4f}")

print("\n=== 4. IF vs LOF (Top-K Jaccard & Overlap) ===")
N = len(df)
print(f"{'Pair':<20} | " + " | ".join([f"Jacc k={k}" for k in k_values]))
print("-" * 80)
for if_run in if_runs:
    jacs = [f"{top_k_jaccard(df[f'{if_run}_score'], df[f'{lof_run}_score'], k):.4f}" for k in k_values]
    print(f"{if_run} vs {lof_run:<8} | " + " | ".join(jacs))

# ==========================================
# PART 3: 4-Group Breakdown (1% Threshold)
# ==========================================
print("\n=== 5. 1% THRESHOLD AGREEMENT MATRIX (IF_003 vs LOF_004) ===")
if_best = 'IF_003'
lof_best = 'LOF_004'

df['Group'] = 'Neither'
df.loc[(df[f'{if_best}_flag'] == 1) & (df[f'{lof_best}_flag'] == 1), 'Group'] = 'Both'
df.loc[(df[f'{if_best}_flag'] == 1) & (df[f'{lof_best}_flag'] == 0), 'Group'] = 'IF_Only'
df.loc[(df[f'{if_best}_flag'] == 0) & (df[f'{lof_best}_flag'] == 1), 'Group'] = 'LOF_Only'

group_summary = df.groupby('Group').agg(
    Total_Count=('row_id', 'count'),
    Fraud_Count=('Class', 'sum')
).reset_index()
group_summary['Fraud_%'] = (group_summary['Fraud_Count'] / group_summary['Total_Count'] * 100).round(2)
group_summary['Recall_Contribution'] = (group_summary['Fraud_Count'] / df['Class'].sum() * 100).round(2)

# Ensure all groups are present
for g in ['Neither', 'IF_Only', 'LOF_Only', 'Both']:
    if g not in group_summary['Group'].values:
        group_summary = pd.concat([group_summary, pd.DataFrame([{'Group': g, 'Total_Count': 0, 'Fraud_Count': 0, 'Fraud_%': 0, 'Recall_Contribution': 0}])], ignore_index=True)

print(group_summary.to_string(index=False))

# ==========================================
# PART 4: Overlap Enrichment (Hypergeometric)
# ==========================================
print("\n=== 6. RANDOM OVERLAP ENRICHMENT ===")
k = 300
observed_overlap = len(df[df['Group'] == 'Both'])
expected_overlap = (k * k) / N
enrichment = observed_overlap / expected_overlap if expected_overlap > 0 else 0

# Hypergeometric test: P(X >= observed_overlap)
# M = total pop (30000), n = successes in pop (top 300 IF), N = draws (top 300 LOF)
p_value = 1 - hypergeom.cdf(observed_overlap - 1, N, k, k)

print(f"Total Observations (N): {N}")
print(f"Flagged by each (k): {k}")
print(f"Expected Random Intersection: {expected_overlap:.2f}")
print(f"Observed Intersection: {observed_overlap}")
print(f"Enrichment Factor: {enrichment:.2f}x")
print(f"Hypergeometric p-value: {p_value:.2e}")

# Save Phase 6 output for visualization later
df.to_csv(os.path.join(STORAGE_DIR, "phase6_group_assignments.csv"), index=False)
print("\nPhase 6 group assignments saved for visualization.")

Loading data for Phase 6...

=== 1. IF SEED STABILITY (Spearman Rank Correlation) ===
IF_001 vs IF_004: 0.9288
IF_001 vs IF_005: 0.9375
IF_004 vs IF_005: 0.9416

=== 2. IF SEED STABILITY (Top-K Jaccard) ===
Pair                 | k=30 | k=150 | k=300 | k=600 | k=1500
----------------------------------------------------------------------
IF_001 vs IF_004       | 0.5789 | 0.6854 | 0.7241 | 0.7857 | 0.7452
IF_001 vs IF_005       | 0.4286 | 0.6575 | 0.6997 | 0.7467 | 0.7251
IF_004 vs IF_005       | 0.4634 | 0.7341 | 0.6901 | 0.7341 | 0.7371

=== 3. IF vs LOF (Spearman Rank Correlation) ===
IF_001 vs LOF_004: 0.3632
IF_003 vs LOF_004: 0.3666
IF_004 vs LOF_004: 0.3773
IF_005 vs LOF_004: 0.3726

=== 4. IF vs LOF (Top-K Jaccard & Overlap) ===
Pair                 | Jacc k=30 | Jacc k=150 | Jacc k=300 | Jacc k=600 | Jacc k=1500
--------------------------------------------------------------------------------
IF_001 vs LOF_004  | 0.1765 | 0.2500 | 0.2220 | 0.1594 | 0.1583
IF_003 vs LOF_004  | 0.2

In [ ]:
# ============================================================
# PHASE 5: Visualization (Canonical Alignment & PCA)
# ============================================================
import os
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from scipy.stats import rankdata

# 1. Define Paths
STORAGE_DIR = "/content/drive/MyDrive/mplad-dvc-storage"
PRED_PATH = os.path.join(STORAGE_DIR, "phase3_canonical_predictions.pkl")
TRUTH_PATH = os.path.join(STORAGE_DIR, "ground_truth_registry.csv")
FS_PATH = os.path.join(STORAGE_DIR, "ULB_FS_A_Baseline.csv")

print("Loading data for Phase 5 visualization...")
with open(PRED_PATH, "rb") as f:
    output_bundle = pickle.load(f)

row_ids = output_bundle['row_ids']
predictions = output_bundle['predictions']
ground_truth = pd.read_csv(TRUTH_PATH)
fs_df = pd.read_csv(FS_PATH)

# 2. Build Canonical DataFrame (Strict Alignment)
df = pd.DataFrame({'row_id': row_ids})
df = df.merge(ground_truth[['row_id', 'Class']], on='row_id', how='left')

# Use IF_003 and LOF_004 as the reference runs (Best post-hoc performers)
df['IF_score'] = -predictions['IF_003']['scores'] # Higher = More anomalous
df['IF_flag'] = predictions['IF_003']['flags']
df['LOF_score'] = -predictions['LOF_004']['scores'] # Higher = More anomalous
df['LOF_flag'] = predictions['LOF_004']['flags']

# Create 4-Group Agreement Categories
df['Group'] = 'Neither'
df.loc[(df['IF_flag'] == 1) & (df['LOF_flag'] == 1), 'Group'] = 'Both'
df.loc[(df['IF_flag'] == 1) & (df['LOF_flag'] == 0), 'Group'] = 'IF_Only'
df.loc[(df['IF_flag'] == 0) & (df['LOF_flag'] == 1), 'Group'] = 'LOF_Only'

# 3. Compute PCA for Visualization Only (Strict Alignment via row_id)
print("Computing 2D PCA for visualization...")

# Merge the features into our main df to guarantee perfect alignment
df = df.merge(fs_df, on='row_id', how='left')

# Extract just the feature columns (V1-V28, Time, Amount) for PCA
feature_cols = [c for c in fs_df.columns if c != 'row_id']
X_30k = df[feature_cols].to_numpy()

pca = PCA(n_components=2, random_state=42)
pc = pca.fit_transform(X_30k)
df['PC1'] = pc[:, 0]
df['PC2'] = pc[:, 1]

print(f"PC1 Explained Variance: {pca.explained_variance_ratio_[0]:.4f}")
print(f"PC2 Explained Variance: {pca.explained_variance_ratio_[1]:.4f}")
print(f"Cumulative: {sum(pca.explained_variance_ratio_):.4f}")

# Convert scores to percentiles (1.0 = most anomalous)
df['IF_pct'] = rankdata(df['IF_score']) / len(df)
df['LOF_pct'] = rankdata(df['LOF_score']) / len(df)

# ==========================================
# VISUALIZATION 1: Rank-vs-Rank Plot
# ==========================================
print("\nGenerating Plot 1: Rank-vs-Rank...")
plt.figure(figsize=(10, 8))
# Background scatter (all points)
plt.scatter(df['IF_pct'], df['LOF_pct'], s=5, c='lightgray', alpha=0.3, label='All Observations')

# Highlight specific groups
both = df[df['Group'] == 'Both']
lof_only_fraud = df[(df['Group'] == 'LOF_Only') & (df['Class'] == 1)]

plt.scatter(both['IF_pct'], both['LOF_pct'], s=30, c='blue', label='Both (107)', edgecolors='black', linewidth=0.5)
plt.scatter(lof_only_fraud['IF_pct'], lof_only_fraud['LOF_pct'], s=50, c='red', marker='X', label='LOF-Only Fraud (8)')

plt.axvline(x=0.99, color='black', linestyle='--', alpha=0.5)
plt.axhline(y=0.99, color='black', linestyle='--', alpha=0.5)

plt.xlabel('IF Anomaly Percentile', fontsize=12)
plt.ylabel('LOF Anomaly Percentile', fontsize=12)
plt.title('Rank-vs-Rank: IF vs LOF Anomaly Percentiles')
plt.legend()
plt.savefig(os.path.join(STORAGE_DIR, "phase5_plot1_rank_vs_rank.png"), dpi=300, bbox_inches='tight')
plt.close()

# ==========================================
# VISUALIZATION 2: PCA Agreement Map
# ==========================================
print("Generating Plot 2: PCA Agreement Map...")
plt.figure(figsize=(10, 8))
# Plot "Neither" first as background
sns.scatterplot(data=df[df['Group'] == 'Neither'], x='PC1', y='PC2', s=5, color='lightgray', alpha=0.3, label='Neither')

# Plot the other groups on top
plot_order = ['IF_Only', 'LOF_Only', 'Both']
colors = ['orange', 'green', 'blue']
for group, color in zip(plot_order, colors):
    subset = df[df['Group'] == group]
    sns.scatterplot(data=subset, x='PC1', y='PC2', s=40, color=color, label=group, edgecolor='black', linewidth=0.5)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.2%})", fontsize=12)
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.2%})", fontsize=12)
plt.title('PCA Agreement Map: IF vs LOF (1% Threshold)')
plt.legend()
plt.savefig(os.path.join(STORAGE_DIR, "phase5_plot2_pca_agreement.png"), dpi=300, bbox_inches='tight')
plt.close()

# ==========================================
# VISUALIZATION 3: Top-K Jaccard Curve
# ==========================================
print("Generating Plot 3: Top-K Jaccard Curve...")
def top_k_jaccard(s1, s2, k):
    top1 = set(np.argsort(s1)[-k:])
    top2 = set(np.argsort(s2)[-k:])
    inter = len(top1 & top2)
    union = len(top1 | top2)
    return inter / union if union > 0 else 0

k_values = [30, 150, 300, 600, 1500]
k_percentages = [k/30000*100 for k in k_values]
jaccards = [top_k_jaccard(df['IF_score'].to_numpy(), df['LOF_score'].to_numpy(), k) for k in k_values]

plt.figure(figsize=(8, 5))
plt.plot(k_percentages, jaccards, marker='o', linestyle='-', color='purple')
plt.xlabel('Top-K Anomaly Percentage (%)', fontsize=12)
plt.ylabel('Jaccard Similarity', fontsize=12)
plt.title('IF vs LOF Agreement across Top-K Thresholds')
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(STORAGE_DIR, "phase5_plot3_jaccard_curve.png"), dpi=300, bbox_inches='tight')
plt.close()

# ==========================================
# VISUALIZATION 4: PCA with Fraud Overlay
# ==========================================
print("Generating Plot 4: PCA with Fraud Overlay...")
plt.figure(figsize=(10, 8))

# Background = Legitimate
legit = df[df['Class'] == 0]
sns.scatterplot(data=legit, x='PC1', y='PC2', s=5, color='lightgray', alpha=0.2, label='Legitimate')

# Highlight Frauds by Group
frauds = df[df['Class'] == 1]
sns.scatterplot(data=frauds[frauds['Group'] == 'Neither'], x='PC1', y='PC2', s=30, color='gray', marker='s', label='Fraud (Neither)', edgecolor='black', linewidth=0.5)
sns.scatterplot(data=frauds[frauds['Group'] == 'IF_Only'], x='PC1', y='PC2', s=40, color='orange', marker='X', label='Fraud (IF_Only)', edgecolor='black', linewidth=0.5)
sns.scatterplot(data=frauds[frauds['Group'] == 'LOF_Only'], x='PC1', y='PC2', s=40, color='green', marker='X', label='Fraud (LOF_Only)', edgecolor='black', linewidth=0.5)
sns.scatterplot(data=frauds[frauds['Group'] == 'Both'], x='PC1', y='PC2', s=50, color='blue', marker='o', label='Fraud (Both)', edgecolor='black', linewidth=0.5)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.2%})", fontsize=12)
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.2%})", fontsize=12)
plt.title('PCA Feature Space: Fraud Overlay by Agreement Group')
plt.legend()
plt.savefig(os.path.join(STORAGE_DIR, "phase5_plot4_pca_fraud_overlay.png"), dpi=300, bbox_inches='tight')
plt.close()

print("\nPhase 5 Complete. All 4 visualizations saved to Google Drive.")

Loading data for Phase 5 visualization...
Computing 2D PCA for visualization...
PC1 Explained Variance: 0.2829
PC2 Explained Variance: 0.0932
Cumulative: 0.3761

Generating Plot 1: Rank-vs-Rank...
Generating Plot 2: PCA Agreement Map...
Generating Plot 3: Top-K Jaccard Curve...
Generating Plot 4: PCA with Fraud Overlay...

Phase 5 Complete. All 4 visualizations saved to Google Drive.


In [ ]:
# ============================================================
# PHASE 5: Expanded Visualization & Diagnostic Analysis
# ============================================================
import os
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics import precision_recall_curve, average_precision_score
from scipy.stats import spearmanr, rankdata

# 1. Define Paths & Load Data
STORAGE_DIR = "/content/drive/MyDrive/mplad-dvc-storage"
PRED_PATH = os.path.join(STORAGE_DIR, "phase3_canonical_predictions.pkl")
TRUTH_PATH = os.path.join(STORAGE_DIR, "ground_truth_registry.csv")
METADATA_PATH = os.path.join(STORAGE_DIR, "phase3_run_metadata.csv")
FS_PATH = os.path.join(STORAGE_DIR, "ULB_FS_A_Baseline.csv")

print("Loading data for Phase 5 diagnostic visualizations...")
with open(PRED_PATH, "rb") as f:
    output_bundle = pickle.load(f)

row_ids = output_bundle['row_ids']
predictions = output_bundle['predictions']
ground_truth = pd.read_csv(TRUTH_PATH)
metadata_df = pd.read_csv(METADATA_PATH)
fs_df = pd.read_csv(FS_PATH)

# 2. Build Canonical DataFrame (Strict Alignment)
df = pd.DataFrame({'row_id': row_ids})
df = df.merge(ground_truth[['row_id', 'Class']], on='row_id', how='left')

# Load ALL 10 runs into the dataframe (Canonical Score: -raw_score so higher=anomalous)
run_ids = list(predictions.keys())
for run_id in run_ids:
    df[f"{run_id}_score"] = -predictions[run_id]['scores']
    df[f"{run_id}_flag"] = predictions[run_id]['flags']

# Compute PCA for Visualization Only
df = df.merge(fs_df, on='row_id', how='left')
feature_cols = [c for c in fs_df.columns if c != 'row_id']
X_30k = df[feature_cols].to_numpy()

pca = PCA(n_components=2, random_state=42)
df['PC1'] = pca.fit_transform(X_30k)[:, 0]
df['PC2'] = pca.fit_transform(X_30k)[:, 1]

# Define Best Runs
if_best = 'IF_003'
lof_best = 'LOF_004'

# Create 4-Group Agreement Categories for best runs
df['Group'] = 'Neither'
df.loc[(df[f'{if_best}_flag'] == 1) & (df[f'{lof_best}_flag'] == 1), 'Group'] = 'Both'
df.loc[(df[f'{if_best}_flag'] == 1) & (df[f'{lof_best}_flag'] == 0), 'Group'] = 'IF_Only'
df.loc[(df[f'{if_best}_flag'] == 0) & (df[f'{lof_best}_flag'] == 1), 'Group'] = 'LOF_Only'

y_true = df['Class'].to_numpy()
base_fraud_rate = y_true.sum() / len(y_true)

# ==========================================
# PLOT 1: Precision-Recall Curves (Supplementary)
# ==========================================
print("Generating Plot 1: PR Curves...")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for run_id in [r for r in run_ids if 'IF' in r]:
    scores = df[f"{run_id}_score"].to_numpy()
    prec, rec, _ = precision_recall_curve(y_true, scores)
    ap = average_precision_score(y_true, scores)
    color = 'red' if run_id == if_best else 'lightcoral'
    lw = 2.5 if run_id == if_best else 1
    axes[0].plot(rec, prec, color=color, lw=lw, label=f'{run_id} (AP={ap:.3f})')
axes[0].set_title('Isolation Forest')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].legend()
axes[0].grid(alpha=0.3)

for run_id in [r for r in run_ids if 'LOF' in r]:
    scores = df[f"{run_id}_score"].to_numpy()
    prec, rec, _ = precision_recall_curve(y_true, scores)
    ap = average_precision_score(y_true, scores)
    color = 'blue' if run_id == lof_best else 'lightskyblue'
    lw = 2.5 if run_id == lof_best else 1
    axes[1].plot(rec, prec, color=color, lw=lw, label=f'{run_id} (AP={ap:.3f})')
axes[1].set_title('Local Outlier Factor')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(STORAGE_DIR, "phase5_plot1_PR_curves.png"), dpi=300)
plt.close()

# ==========================================
# PLOT 2: Fraud Discovery / Review-Budget Curve (Main)
# ==========================================
print("Generating Plot 2: Fraud Discovery Curve...")
plt.figure(figsize=(10, 6))
percentages = np.linspace(0.001, 0.1, 100) # 0.1% to 10%
total_frauds = y_true.sum()

for run_id, color, label in [(if_best, 'red', 'IF_003'), (lof_best, 'blue', 'LOF_004')]:
    scores = df[f"{run_id}_score"].to_numpy()
    sorted_idx = np.argsort(scores)[::-1] # Highest score first
    y_sorted = y_true[sorted_idx]

    recalls = []
    for p in percentages:
        k = int(len(y_sorted) * p)
        recalls.append(y_sorted[:k].sum() / total_frauds)

    plt.plot(percentages * 100, recalls, marker='o', linestyle='-', color=color, label=label)

# Random baseline
random_recalls = percentages * len(y_true) / total_frauds
plt.plot(percentages * 100, random_recalls, linestyle='--', color='gray', label='Random Selection')

plt.xlabel('Percentage of Transactions Investigated (%)', fontsize=12)
plt.ylabel('Cumulative Fraud Recall', fontsize=12)
plt.title('Fraud Discovery vs Review Budget')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(os.path.join(STORAGE_DIR, "phase5_plot2_fraud_discovery.png"), dpi=300)
plt.close()

# ==========================================
# PLOT 3: Anomaly-Score ECDF by Class (Supplementary)
# ==========================================
print("Generating Plot 3: Score ECDF by Class...")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.ecdfplot(data=df, x=f"{if_best}_score", hue="Class", ax=axes[0], palette={0: 'gray', 1: 'red'})
axes[0].set_title(f'IF Anomaly Score ECDF ({if_best})')
axes[0].set_xlabel('Canonical Anomaly Score (Higher = More Anomalous)')
axes[0].grid(alpha=0.3)

sns.ecdfplot(data=df, x=f"{lof_best}_score", hue="Class", ax=axes[1], palette={0: 'gray', 1: 'blue'})
axes[1].set_title(f'LOF Anomaly Score ECDF ({lof_best})')
axes[1].set_xlabel('Canonical Anomaly Score (Higher = More Anomalous)')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(STORAGE_DIR, "phase5_plot3_ECDF.png"), dpi=300)
plt.close()

# ==========================================
# PLOT 4: IF-vs-LOF Rank Plot (Main)
# ==========================================
print("Generating Plot 4: Rank-vs-Rank Plot...")
df['IF_pct'] = rankdata(df[f"{if_best}_score"]) / len(df)
df['LOF_pct'] = rankdata(df[f"{lof_best}_score"]) / len(df)

plt.figure(figsize=(9, 9))
plt.scatter(df['IF_pct'], df['LOF_pct'], s=5, c='lightgray', alpha=0.3, label='Neither')

both = df[df['Group'] == 'Both']
lof_only_fraud = df[(df['Group'] == 'LOF_Only') & (df['Class'] == 1)]

plt.scatter(both['IF_pct'], both['LOF_pct'], s=30, c='blue', label='Both', edgecolors='black', linewidth=0.5)
plt.scatter(lof_only_fraud['IF_pct'], lof_only_fraud['LOF_pct'], s=80, c='red', marker='X', label='LOF-Only Fraud')

plt.axvline(x=0.99, color='black', linestyle='--', alpha=0.5)
plt.axhline(y=0.99, color='black', linestyle='--', alpha=0.5)
plt.text(0.95, 0.95, 'Top 1%', fontsize=12, fontweight='bold')

corr, _ = spearmanr(df[f"{if_best}_score"], df[f"{lof_best}_score"])
plt.text(0.05, 0.95, f'Spearman ρ = {corr:.4f}', fontsize=14, bbox=dict(facecolor='white', alpha=0.8))

plt.xlabel('IF Anomaly Percentile', fontsize=13)
plt.ylabel('LOF Anomaly Percentile', fontsize=13)
plt.title('Cross-Method Rank Agreement (IF vs LOF)')
plt.legend()
plt.savefig(os.path.join(STORAGE_DIR, "phase5_plot4_rank_vs_rank.png"), dpi=300)
plt.close()

# ==========================================
# PLOT 5: Fraud Enrichment / Lift Plot (Main)
# ==========================================
print("Generating Plot 5: Fraud Enrichment (Lift) Plot...")
group_lift = df.groupby('Group').apply(lambda x: (x['Class'].sum() / len(x)) / base_fraud_rate).reset_index()
group_lift.columns = ['Group', 'Lift']
group_lift = group_lift.sort_values('Lift', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=group_lift, x='Group', y='Lift', palette=['blue', 'green', 'orange', 'gray'])
plt.yscale('log')
plt.title('Fraud Enrichment (Lift) by Agreement Group')
plt.ylabel('Lift vs Baseline (Log Scale)')
plt.xlabel('Agreement Group')
plt.savefig(os.path.join(STORAGE_DIR, "phase5_plot5_enrichment_lift.png"), dpi=300)
plt.close()

# ==========================================
# PLOT 6: 10x10 Spearman Correlation Matrix (Supplementary)
# ==========================================
print("Generating Plot 6: 10x10 Run Correlation Matrix...")
score_cols = [f"{r}_score" for r in run_ids]
corr_matrix = df[score_cols].corr(method='spearman')

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=0, vmax=1, fmt=".2f")
plt.title('Spearman Rank Correlation Across All 10 Runs')
plt.savefig(os.path.join(STORAGE_DIR, "phase5_plot6_corr_matrix.png"), dpi=300)
plt.close()

# ==========================================
# PLOT 7: LOF Neighborhood Sensitivity (Main)
# ==========================================
print("Generating Plot 7: LOF Sensitivity Curve...")
lof_metrics = metadata_df[metadata_df['model'] == 'LOF'].merge(
    pd.DataFrame([
        {'run_id': 'LOF_001', 'k': 5}, {'run_id': 'LOF_002', 'k': 15},
        {'run_id': 'LOF_003', 'k': 30}, {'run_id': 'LOF_004', 'k': 50},
        {'run_id': 'LOF_005', 'k': 30} # Manhattan
    ]), on='run_id'
)
# Filter out manhattan for the k-sensitivity plot
lof_euc = lof_metrics[lof_metrics['metric'] == 'euclidean']

# Merge with Phase 4 AP (calculated inline for simplicity)
aps = []
for rid in lof_euc['run_id']:
    aps.append(average_precision_score(y_true, df[f"{rid}_score"].to_numpy()))
lof_euc['AP'] = aps

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.lineplot(data=lof_euc, x='k', y='AP', marker='o', ax=axes[0], color='blue')
axes[0].set_title('LOF Average Precision vs Neighborhood Size')
axes[0].set_xlabel('n_neighbors (k)')
axes[0].set_ylabel('Average Precision (AP)')
axes[0].grid(alpha=0.3)

sns.lineplot(data=lof_euc, x='k', y='runtime_seconds', marker='o', ax=axes[1], color='red')
axes[1].set_title('LOF Execution Time vs Neighborhood Size')
axes[1].set_xlabel('n_neighbors (k)')
axes[1].set_ylabel('Runtime (seconds)')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(STORAGE_DIR, "phase5_plot7_lof_sensitivity.png"), dpi=300)
plt.close()

# ==========================================
# PLOT 8: PCA Agreement Map (Flagged Only) (Main)
# ==========================================
print("Generating Plot 8: PCA Agreement Map (Flagged Only)...")
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: Full Population faintly
axes[0].scatter(df['PC1'], df['PC2'], s=2, c='lightgray', alpha=0.2)
for group, color in [('Both', 'blue'), ('IF_Only', 'orange'), ('LOF_Only', 'green')]:
    subset = df[df['Group'] == group]
    axes[0].scatter(subset['PC1'], subset['PC2'], s=40, c=color, label=group, edgecolor='black', linewidth=0.5)
axes[0].set_title('Full Population + Flagged Overlays')
axes[0].legend()

# Right: Flagged Population Only
for group, color in [('Both', 'blue'), ('IF_Only', 'orange'), ('LOF_Only', 'green')]:
    subset = df[df['Group'] == group]
    axes[1].scatter(subset['PC1'], subset['PC2'], s=50, c=color, label=group, edgecolor='black', linewidth=0.5)
axes[1].set_title('Flagged Population Only (600 obs)')

plt.suptitle('PCA Feature Space: Agreement Groups', fontsize=16)
plt.savefig(os.path.join(STORAGE_DIR, "phase5_plot8_pca_agreement.png"), dpi=300)
plt.close()

print("\nPhase 5 Complete. 8 high-resolution diagnostic plots saved to Google Drive.")

Loading data for Phase 5 diagnostic visualizations...
Generating Plot 1: PR Curves...
Generating Plot 2: Fraud Discovery Curve...
Generating Plot 3: Score ECDF by Class...
Generating Plot 4: Rank-vs-Rank Plot...
Generating Plot 5: Fraud Enrichment (Lift) Plot...


/tmp/ipykernel_2648/302103879.py:181: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  group_lift = df.groupby('Group').apply(lambda x: (x['Class'].sum() / len(x)) / base_fraud_rate).reset_index()
/tmp/ipykernel_2648/302103879.py:186: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=group_lift, x='Group', y='Lift', palette=['blue', 'green', 'orange', 'gray'])


Generating Plot 6: 10x10 Run Correlation Matrix...
Generating Plot 7: LOF Sensitivity Curve...


/tmp/ipykernel_2648/302103879.py:225: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lof_euc['AP'] = aps


Generating Plot 8: PCA Agreement Map (Flagged Only)...

Phase 5 Complete. 8 high-resolution diagnostic plots saved to Google Drive.


In [ ]:
# ============================================================
# PHASE 5: Expanded Quantitative & Diagnostic Analysis
# ============================================================
import os
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_auc_score, precision_score, recall_score, f1_score
from scipy.stats import spearmanr, rankdata, hypergeom, ks_2samp
from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')

# 1. Define Paths & Load Data
STORAGE_DIR = "/content/drive/MyDrive/mplad-dvc-storage"
PRED_PATH = os.path.join(STORAGE_DIR, "phase3_canonical_predictions.pkl")
TRUTH_PATH = os.path.join(STORAGE_DIR, "ground_truth_registry.csv")
METADATA_PATH = os.path.join(STORAGE_DIR, "phase3_run_metadata.csv")
FS_PATH = os.path.join(STORAGE_DIR, "ULB_FS_A_Baseline.csv")

print("Loading data for Phase 5 diagnostic analysis...")
with open(PRED_PATH, "rb") as f:
    output_bundle = pickle.load(f)

row_ids = output_bundle['row_ids']
predictions = output_bundle['predictions']
ground_truth = pd.read_csv(TRUTH_PATH)
metadata_df = pd.read_csv(METADATA_PATH)
fs_df = pd.read_csv(FS_PATH)

# 2. Build Canonical DataFrame
df = pd.DataFrame({'row_id': row_ids})
df = df.merge(ground_truth[['row_id', 'Class']], on='row_id', how='left')

run_ids = list(predictions.keys())
for run_id in run_ids:
    df[f"{run_id}_score"] = -predictions[run_id]['scores'] # Canonical: Higher = anomalous
    df[f"{run_id}_flag"] = predictions[run_id]['flags']

y_true = df['Class'].to_numpy()
base_fraud_rate = y_true.sum() / len(y_true)
if_best = 'IF_003'
lof_best = 'LOF_004'

# ==========================================
# COMPUTATION 1: Stratified Bootstrap CIs
# ==========================================
print("\n=== 1. STRATIFIED BOOTSTRAP CONFIDENCE INTERVALS (2000 iterations) ===")
def bootstrap_metrics(y, scores, n_iters=2000):
    fraud_idx = np.where(y == 1)[0]
    legit_idx = np.where(y == 0)[0]
    aps, rocs, precs, recs, f1s = [], [], [], [], []

    for _ in range(n_iters):
        boot_fraud = np.random.choice(fraud_idx, len(fraud_idx), replace=True)
        boot_legit = np.random.choice(legit_idx, len(legit_idx), replace=True)
        boot_idx = np.concatenate([boot_fraud, boot_legit])

        y_boot = y[boot_idx]
        s_boot = scores[boot_idx]

        aps.append(average_precision_score(y_boot, s_boot))
        rocs.append(roc_auc_score(y_boot, s_boot))

        thresh = np.percentile(s_boot, 99) # Top 1%
        flags = np.where(s_boot >= thresh, 1, 0)
        precs.append(precision_score(y_boot, flags, zero_division=0))
        recs.append(recall_score(y_boot, flags, zero_division=0))
        f1s.append(f1_score(y_boot, flags, zero_division=0))

    return {
        'AP': (round(np.percentile(aps, 2.5), 4), round(np.percentile(aps, 97.5), 4)),
        'ROC': (round(np.percentile(rocs, 2.5), 4), round(np.percentile(rocs, 97.5), 4)),
        'Prec': (round(np.percentile(precs, 2.5), 4), round(np.percentile(precs, 97.5), 4)),
        'Rec': (round(np.percentile(recs, 2.5), 4), round(np.percentile(recs, 97.5), 4)),
        'F1': (round(np.percentile(f1s, 2.5), 4), round(np.percentile(f1s, 97.5), 4))
    }

bootstrap_results = {}
for run_id in [if_best, 'IF_004', lof_best]:
    bootstrap_results[run_id] = bootstrap_metrics(y_true, df[f"{run_id}_score"].to_numpy())
    print(f"{run_id}: {bootstrap_results[run_id]}")

# ==========================================
# COMPUTATION 2: Hypergeometric Overlap Test
# ==========================================
print("\n=== 2. HYPERGEOMETRIC OVERLAP TEST ===")
N = 30000
K = 300
n = 300
observed_overlap = len(df[(df[f"{if_best}_flag"]==1) & (df[f"{lof_best}_flag"]==1)])

expected = n * K / N
p_value = hypergeom.sf(observed_overlap - 1, N, K, n)
log10_p = hypergeom.logsf(observed_overlap - 1, N, K, n) / np.log(10)
print(f"Expected random overlap: {expected:.2f}")
print(f"Observed overlap: {observed_overlap}")
print(f"Enrichment Factor: {observed_overlap/expected:.2f}x")
print(f"P-value: < 10^{log10_p:.2f}")

# ==========================================
# COMPUTATION 3: 10x10 Spearman & Jaccard Matrices
# ==========================================
print("\n=== 3. BUILDING 10x10 STABILITY MATRICES ===")
score_cols = [f"{r}_score" for r in run_ids]
spearman_matrix = df[score_cols].corr(method='spearman')

# Jaccard @ 1% (300 obs)
jaccard_matrix = pd.DataFrame(index=run_ids, columns=run_ids, dtype=float)
top_k_sets = {r: set(np.argsort(df[f"{r}_score"].to_numpy())[-300:]) for r in run_ids}

for r1 in run_ids:
    for r2 in run_ids:
        inter = len(top_k_sets[r1] & top_k_sets[r2])
        union = len(top_k_sets[r1] | top_k_sets[r2])
        jaccard_matrix.loc[r1, r2] = inter / union if union > 0 else 0

# ==========================================
# COMPUTATION 4: Multi-Threshold Group Analysis
# ==========================================
print("\n=== 4. MULTI-THRESHOLD AGREEMENT ANALYSIS ===")
k_vals = [30, 75, 150, 300, 600, 1500] # 0.1% to 5%
multi_thresh_records = []

if_scores = df[f"{if_best}_score"].to_numpy()
lof_scores = df[f"{lof_best}_score"].to_numpy()

for k in k_vals:
    pct = k / 30000 * 100
    top_if = set(np.argsort(if_scores)[-k:])
    top_lof = set(np.argsort(lof_scores)[-k:])

    both = top_if & top_lof
    if_only = top_if - top_lof
    lof_only = top_lof - top_if

    def get_fraud_count(id_set):
        return df.iloc[list(id_set)]['Class'].sum()

    multi_thresh_records.append({
        'Percentile': f"{pct:.1f}%",
        'Both_Count': len(both), 'Both_Fraud': get_fraud_count(both),
        'IF_Only_Count': len(if_only), 'IF_Only_Fraud': get_fraud_count(if_only),
        'LOF_Only_Count': len(lof_only), 'LOF_Only_Fraud': get_fraud_count(lof_only)
    })

multi_thresh_df = pd.DataFrame(multi_thresh_records)
print(multi_thresh_df.to_string(index=False))

# ==========================================
# VISUALIZATIONS
# ==========================================
print("\nGenerating Diagnostic Visualizations...")

# PLOT 1: PR Curves (Best vs Rest)
plt.figure(figsize=(10, 6))
for run_id in run_ids:
    scores = df[f"{run_id}_score"].to_numpy()
    prec, rec, _ = precision_recall_curve(y_true, scores)
    if run_id == if_best:
        plt.plot(rec, prec, color='red', lw=2.5, label=f'{run_id} (Best IF)')
    elif run_id == lof_best:
        plt.plot(rec, prec, color='blue', lw=2.5, label=f'{run_id} (Best LOF)')
    elif 'IF' in run_id:
        plt.plot(rec, prec, color='red', lw=1, alpha=0.5)
    else:
        plt.plot(rec, prec, color='blue', lw=1, alpha=0.5)
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curves (IF vs LOF)')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(STORAGE_DIR, "phase5_plot1_PR_curves.png"), dpi=300); plt.close()

# PLOT 2: Fraud Discovery & Precision vs Budget
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
percentages = np.linspace(0.001, 0.1, 100) # 0.1% to 10%

for run_id, color in [(if_best, 'red'), (lof_best, 'blue')]:
    scores = df[f"{run_id}_score"].to_numpy()
    sorted_idx = np.argsort(scores)[::-1]
    y_sorted = y_true[sorted_idx]

    recalls, precs = [], []
    for p in percentages:
        k = int(len(y_sorted) * p)
        recalls.append(y_sorted[:k].sum() / y_true.sum())
        precs.append(y_sorted[:k].sum() / k)

    axes[0].plot(percentages*100, recalls, color=color, label=run_id)
    axes[1].plot(percentages*100, precs, color=color, label=run_id)

axes[0].plot(percentages*100, percentages, linestyle='--', color='gray', label='Random')
axes[1].axhline(y=base_fraud_rate, linestyle='--', color='gray', label='Base Rate')

axes[0].set_title('Fraud Discovery (Recall vs Review Budget)')
axes[0].set_xlabel('% of Data Reviewed'); axes[0].set_ylabel('Recall')
axes[1].set_title('Precision vs Review Budget')
axes[1].set_xlabel('% of Data Reviewed'); axes[1].set_ylabel('Precision')
axes[0].legend(); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(STORAGE_DIR, "phase5_plot2_fraud_discovery_precision.png"), dpi=300); plt.close()

# PLOT 3: Hypergeometric Null Distribution
plt.figure(figsize=(8, 5))
x = np.arange(0, 30)
pmf = hypergeom.pmf(x, N, K, n)
plt.bar(x, pmf, color='gray', alpha=0.5, label='Random Null Distribution')
plt.axvline(x=observed_overlap, color='red', linestyle='--', lw=2, label=f'Observed Overlap ({observed_overlap})')
plt.title('IF vs LOF Overlap vs Random Null (Hypergeometric)')
plt.xlabel('Intersection Size'); plt.ylabel('Probability')
plt.legend()
plt.savefig(os.path.join(STORAGE_DIR, "phase5_plot3_hypergeom.png"), dpi=300); plt.close()

# PLOT 4: 10x10 Spearman Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(spearman_matrix.astype(float), annot=True, cmap='coolwarm', vmin=0, vmax=1, fmt=".2f")
plt.title('10x10 Spearman Rank Correlation Matrix')
plt.savefig(os.path.join(STORAGE_DIR, "phase5_plot4_spearman_matrix.png"), dpi=300); plt.close()

# PLOT 5: 10x10 Jaccard @ 1% Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(jaccard_matrix.astype(float), annot=True, cmap='Blues', vmin=0, vmax=1, fmt=".2f")
plt.title('10x10 Top-1% Jaccard Similarity Matrix')
plt.savefig(os.path.join(STORAGE_DIR, "phase5_plot5_jaccard_matrix.png"), dpi=300); plt.close()

# PLOT 6: Top-K Jaccard Curve
k_perc = [k/30000*100 for k in k_vals]
jacs = []
for k in k_vals:
    top_if = set(np.argsort(if_scores)[-k:])
    top_lof = set(np.argsort(lof_scores)[-k:])
    inter = len(top_if & top_lof)
    union = len(top_if | top_lof)
    jacs.append(inter/union if union > 0 else 0)

plt.figure(figsize=(8, 5))
plt.plot(k_perc, jacs, marker='o', color='purple')
plt.xlabel('Top-K Percentage (%)'); plt.ylabel('Jaccard Similarity')
plt.title('IF vs LOF Agreement across Top-K Thresholds')
plt.grid(alpha=0.3)
plt.savefig(os.path.join(STORAGE_DIR, "phase5_plot6_topk_jaccard.png"), dpi=300); plt.close()

# PLOT 7: PCA Agreement Map (Flagged Only)
df = df.merge(fs_df, on='row_id', how='left')
feature_cols = [c for c in fs_df.columns if c != 'row_id']
pca = PCA(n_components=2, random_state=42)
df['PC1'], df['PC2'] = pca.fit_transform(df[feature_cols].to_numpy()).T

df['Group'] = 'Neither'
df.loc[(df[f'{if_best}_flag'] == 1) & (df[f'{lof_best}_flag'] == 1), 'Group'] = 'Both'
df.loc[(df[f'{if_best}_flag'] == 1) & (df[f'{lof_best}_flag'] == 0), 'Group'] = 'IF_Only'
df.loc[(df[f'{if_best}_flag'] == 0) & (df[f'{lof_best}_flag'] == 1), 'Group'] = 'LOF_Only'

plt.figure(figsize=(10, 8))
# Background
plt.scatter(df[df['Group']=='Neither']['PC1'], df[df['Group']=='Neither']['PC2'], s=2, c='lightgray', alpha=0.2)
# Flagged
for group, color in [('IF_Only', 'orange'), ('LOF_Only', 'green'), ('Both', 'blue')]:
    subset = df[df['Group'] == group]
    plt.scatter(subset['PC1'], subset['PC2'], s=40, c=color, label=group, edgecolor='black', linewidth=0.5)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.2%})")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.2%})")
plt.title('PCA Agreement Map (Flagged Overlays)')
plt.legend()
plt.savefig(os.path.join(STORAGE_DIR, "phase5_plot7_pca_agreement.png"), dpi=300); plt.close()

# PLOT 8: Rank-vs-Rank Plot
df['IF_pct'] = rankdata(df[f"{if_best}_score"]) / len(df)
df['LOF_pct'] = rankdata(df[f"{lof_best}_score"]) / len(df)

plt.figure(figsize=(9, 9))
plt.scatter(df['IF_pct'], df['LOF_pct'], s=5, c='lightgray', alpha=0.3, label='Neither')
both = df[df['Group'] == 'Both']
lof_only_fraud = df[(df['Group'] == 'LOF_Only') & (df['Class'] == 1)]
plt.scatter(both['IF_pct'], both['LOF_pct'], s=30, c='blue', label='Both', edgecolors='black', linewidth=0.5)
plt.scatter(lof_only_fraud['IF_pct'], lof_only_fraud['LOF_pct'], s=80, c='red', marker='X', label='LOF-Only Fraud')
plt.axvline(x=0.99, color='black', linestyle='--', alpha=0.5)
plt.axhline(y=0.99, color='black', linestyle='--', alpha=0.5)
plt.xlabel('IF Anomaly Percentile'); plt.ylabel('LOF Anomaly Percentile')
plt.title('Rank-vs-Rank: IF vs LOF')
plt.legend()
plt.savefig(os.path.join(STORAGE_DIR, "phase5_plot8_rank_vs_rank.png"), dpi=300); plt.close()

print("\nPhase 5 Complete. 8 diagnostic plots and quantitative analyses saved to Drive.")

Loading data for Phase 5 diagnostic analysis...

=== 1. STRATIFIED BOOTSTRAP CONFIDENCE INTERVALS (2000 iterations) ===
IF_003: {'AP': (np.float64(0.1981), np.float64(0.4725)), 'ROC': (np.float64(0.9682), np.float64(0.9901)), 'Prec': (np.float64(0.0797), np.float64(0.1229)), 'Rec': (np.float64(0.5106), np.float64(0.7872)), 'F1': (np.float64(0.1379), np.float64(0.2126))}
IF_004: {'AP': (np.float64(0.136), np.float64(0.3472)), 'ROC': (np.float64(0.9655), np.float64(0.9903)), 'Prec': (np.float64(0.0833), np.float64(0.1267)), 'Rec': (np.float64(0.5319), np.float64(0.8085)), 'F1': (np.float64(0.1441), np.float64(0.219))}
LOF_004: {'AP': (np.float64(0.2291), np.float64(0.4732)), 'ROC': (np.float64(0.9085), np.float64(0.9852)), 'Prec': (np.float64(0.1067), np.float64(0.1429)), 'Rec': (np.float64(0.6809), np.float64(0.9149)), 'F1': (np.float64(0.1844), np.float64(0.2471))}

=== 2. HYPERGEOMETRIC OVERLAP TEST ===
Expected random overlap: 3.00
Observed overlap: 107
Enrichment Factor: 35.67x
P-va

In [ ]:
# ============================================================
# COMPLETE PIPELINE: Phase 3, 4, 5, & 6 (Full 284k Dataset)
# ============================================================
import os
import pandas as pd
import numpy as np
import time
import pickle
import gc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.decomposition import PCA
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
from scipy.stats import spearmanr, rankdata, hypergeom
import warnings
warnings.filterwarnings('ignore')

STORAGE_DIR = "/content/drive/MyDrive/mplad-dvc-storage"
FS_PATH = os.path.join(STORAGE_DIR, "ULB_FS_A_Baseline.csv")
TRUTH_PATH = os.path.join(STORAGE_DIR, "ground_truth_registry.csv")

print("Loading FULL dataset (284,807 rows)...")
fs_df = pd.read_csv(FS_PATH)
truth_df = pd.read_csv(TRUTH_PATH)

row_ids = fs_df['row_id'].to_numpy()
X = fs_df.drop(columns=['row_id']).to_numpy()
N = len(X)

# Merge ground truth immediately
df = pd.DataFrame({'row_id': row_ids})
df = df.merge(truth_df[['row_id', 'Class']], on='row_id', how='left')
y_true = df['Class'].to_numpy()

# For LOF, take a random 50k unsupervised subsample to fit (prevents Colab deadlock)
np.random.seed(42)
sample_indices = np.random.choice(N, size=50000, replace=False)
X_lof_sample = X[sample_indices]

THRESHOLD_PERCENTILE = 1.0
run_metadata = []
canonical_predictions = {}

# ==========================================
# PHASE 3: MODELING
# ==========================================
print("\n--- PHASE 3: MODELING ---")

# DETECTOR 1: Isolation Forest (Full Data)
print("Running Isolation Forest (Full Data)...")
if_configs = [
    {'n_estimators': 100, 'max_samples': 256, 'random_state': 42},
    {'n_estimators': 200, 'max_samples': 256, 'random_state': 42},
    {'n_estimators': 100, 'max_samples': 0.05, 'random_state': 42},
    {'n_estimators': 100, 'max_samples': 256, 'random_state': 43},
    {'n_estimators': 100, 'max_samples': 256, 'random_state': 44}
]

for i, cfg in enumerate(if_configs):
    run_id = f"IF_{i+1:03d}"
    print(f"  {run_id} ...", end=" ", flush=True)
    start = time.time()

    iso = IsolationForest(n_estimators=cfg['n_estimators'], max_samples=cfg['max_samples'],
                          random_state=cfg['random_state'], n_jobs=2)
    iso.fit(X)
    raw_scores = iso.decision_function(X)
    exec_t = time.time() - start

    threshold_val = np.percentile(raw_scores, THRESHOLD_PERCENTILE)
    flags = np.where(raw_scores <= threshold_val, 1, 0).astype(np.uint8)
    scores_f32 = raw_scores.astype(np.float32)

    run_metadata.append({'run_id': run_id, 'model': 'IsolationForest', 'n_estimators': cfg['n_estimators'],
                         'max_samples': cfg['max_samples'], 'random_state': cfg['random_state'], 'runtime_seconds': round(exec_t, 2)})
    canonical_predictions[run_id] = {'scores': scores_f32, 'flags': flags}
    print(f"Done in {exec_t:.2f}s")
    del iso, raw_scores, flags, scores_f32
    gc.collect()

# DETECTOR 2: Local Outlier Factor (Novelty Mode: Fit 50k, Predict 284k)
print("\nRunning LOF (Novelty Mode: Fit 50k, Predict 284k)...")
lof_configs = [
    {'n_neighbors': 5, 'metric': 'euclidean'},
    {'n_neighbors': 15, 'metric': 'euclidean'},
    {'n_neighbors': 30, 'metric': 'euclidean'},
    {'n_neighbors': 50, 'metric': 'euclidean'},
    {'n_neighbors': 30, 'metric': 'manhattan'}
]

for i, cfg in enumerate(lof_configs):
    run_id = f"LOF_{i+1:03d}"
    print(f"  {run_id} ...", end=" ", flush=True)
    start = time.time()

    lof = LocalOutlierFactor(n_neighbors=cfg['n_neighbors'], metric=cfg['metric'],
                             algorithm='ball_tree', leaf_size=40, n_jobs=2, novelty=True)
    lof.fit(X_lof_sample)
    raw_scores = lof.score_samples(X) # Predict on all 284k
    exec_t = time.time() - start

    threshold_val = np.percentile(raw_scores, THRESHOLD_PERCENTILE)
    flags = np.where(raw_scores <= threshold_val, 1, 0).astype(np.uint8)
    scores_f32 = raw_scores.astype(np.float32)

    run_metadata.append({'run_id': run_id, 'model': 'LOF', 'n_neighbors': cfg['n_neighbors'],
                         'metric': cfg['metric'], 'random_state': 'N/A', 'runtime_seconds': round(exec_t, 2)})
    canonical_predictions[run_id] = {'scores': scores_f32, 'flags': flags}
    print(f"Done in {exec_t:.2f}s")
    del lof, raw_scores, flags, scores_f32
    gc.collect()

metadata_df = pd.DataFrame(run_metadata)
run_ids = list(canonical_predictions.keys())

# Load scores into main df
for run_id in run_ids:
    df[f"{run_id}_score"] = -canonical_predictions[run_id]['scores'] # Canonical: Higher = anomalous
    df[f"{run_id}_flag"] = canonical_predictions[run_id]['flags']

if_best = 'IF_003'
lof_best = 'LOF_004'

# ==========================================
# PHASE 4: VALIDATION
# ==========================================
print("\n--- PHASE 4: VALIDATION ---")
eval_results = []
for run_id in run_ids:
    scores = df[f"{run_id}_score"].to_numpy()
    flags = df[f"{run_id}_flag"].to_numpy()

    ap = average_precision_score(y_true, scores)
    roc = roc_auc_score(y_true, scores)
    prec = precision_score(y_true, flags, zero_division=0)
    rec = recall_score(y_true, flags, zero_division=0)
    f1 = f1_score(y_true, flags, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, flags, labels=[0, 1]).ravel()

    meta = metadata_df[metadata_df['run_id'] == run_id].iloc[0]
    eval_results.append({'run_id': run_id, 'model': meta['model'], 'AP': round(ap,4), 'ROC_AUC': round(roc,4),
                         'Prec': round(prec,4), 'Rec': round(rec,4), 'F1': round(f1,4), 'TP': tp, 'FP': fp})

eval_df = pd.DataFrame(eval_results)
print(eval_df.to_string(index=False))

# ==========================================
# PHASE 5 & 6: DIAGNOSTICS & CONSISTENCY
# ==========================================
print("\n--- PHASE 5 & 6: DIAGNOSTICS & CONSISTENCY ---")

# 1. Agreement Groups
df['Group'] = 'Neither'
df.loc[(df[f'{if_best}_flag'] == 1) & (df[f'{lof_best}_flag'] == 1), 'Group'] = 'Both'
df.loc[(df[f'{if_best}_flag'] == 1) & (df[f'{lof_best}_flag'] == 0), 'Group'] = 'IF_Only'
df.loc[(df[f'{if_best}_flag'] == 0) & (df[f'{lof_best}_flag'] == 1), 'Group'] = 'LOF_Only'

# 2. Hypergeometric Test
k = int(N * 0.01)
observed_overlap = len(df[df['Group'] == 'Both'])
expected = (k * k) / N
p_value = hypergeom.sf(observed_overlap - 1, N, k, k)
log10_p = hypergeom.logsf(observed_overlap - 1, N, k, k) / np.log(10)
print(f"Hypergeometric Overlap: Expected={expected:.2f}, Observed={observed_overlap}, Enrichment={observed_overlap/expected:.2f}x, p<10^{log10_p:.2f}")

# 3. 10x10 Matrices
score_cols = [f"{r}_score" for r in run_ids]
spearman_matrix = df[score_cols].corr(method='spearman')
jaccard_matrix = pd.DataFrame(index=run_ids, columns=run_ids, dtype=float)
top_k_sets = {r: set(np.argsort(df[f"{r}_score"].to_numpy())[-k:]) for r in run_ids}
for r1 in run_ids:
    for r2 in run_ids:
        inter = len(top_k_sets[r1] & top_k_sets[r2])
        union = len(top_k_sets[r1] | top_k_sets[r2])
        jaccard_matrix.loc[r1, r2] = inter / union if union > 0 else 0

# ==========================================
# VISUALIZATIONS
# ==========================================
print("\nGenerating Visualizations...")

# PLOT 1: PR Curves
plt.figure(figsize=(10, 6))
for run_id in run_ids:
    scores = df[f"{run_id}_score"].to_numpy()
    prec_arr, rec_arr, _ = precision_recall_curve(y_true, scores)
    if run_id == if_best: plt.plot(rec_arr, prec_arr, color='red', lw=2.5, label=f'{run_id} (Best IF)')
    elif run_id == lof_best: plt.plot(rec_arr, prec_arr, color='blue', lw=2.5, label=f'{run_id} (Best LOF)')
    elif 'IF' in run_id: plt.plot(rec_arr, prec_arr, color='red', lw=1, alpha=0.5)
    else: plt.plot(rec_arr, prec_arr, color='blue', lw=1, alpha=0.5)
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curves (Full Dataset)')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(STORAGE_DIR, "FULL_plot1_PR_curves.png"), dpi=300); plt.close()

# PLOT 2: Fraud Discovery
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
percentages = np.linspace(0.001, 0.1, 100)
for run_id, color in [(if_best, 'red'), (lof_best, 'blue')]:
    scores = df[f"{run_id}_score"].to_numpy()
    sorted_idx = np.argsort(scores)[::-1]
    y_sorted = y_true[sorted_idx]
    recalls, precs = [], []
    for p in percentages:
        k_val = int(len(y_sorted) * p)
        recalls.append(y_sorted[:k_val].sum() / y_true.sum())
        precs.append(y_sorted[:k_val].sum() / k_val)
    axes[0].plot(percentages*100, recalls, color=color, label=run_id)
    axes[1].plot(percentages*100, precs, color=color, label=run_id)
axes[0].plot(percentages*100, percentages, linestyle='--', color='gray', label='Random')
axes[1].axhline(y=y_true.sum()/N, linestyle='--', color='gray', label='Base Rate')
axes[0].set_title('Fraud Discovery (Recall vs Budget)'); axes[0].legend()
axes[1].set_title('Precision vs Budget'); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(STORAGE_DIR, "FULL_plot2_fraud_discovery.png"), dpi=300); plt.close()

# PLOT 3: Hypergeometric Null
plt.figure(figsize=(8, 5))
x = np.arange(0, 50)
pmf = hypergeom.pmf(x, N, k, k)
plt.bar(x, pmf, color='gray', alpha=0.5, label='Random Null')
plt.axvline(x=observed_overlap, color='red', linestyle='--', lw=2, label=f'Observed ({observed_overlap})')
plt.title('IF vs LOF Overlap vs Random Null (Full Data)')
plt.legend()
plt.savefig(os.path.join(STORAGE_DIR, "FULL_plot3_hypergeom.png"), dpi=300); plt.close()

# PLOT 4 & 5: 10x10 Heatmaps
plt.figure(figsize=(10, 8))
sns.heatmap(spearman_matrix.astype(float), annot=True, cmap='coolwarm', vmin=0, vmax=1, fmt=".2f")
plt.title('10x10 Spearman Rank Correlation')
plt.savefig(os.path.join(STORAGE_DIR, "FULL_plot4_spearman_matrix.png"), dpi=300); plt.close()

plt.figure(figsize=(10, 8))
sns.heatmap(jaccard_matrix.astype(float), annot=True, cmap='Blues', vmin=0, vmax=1, fmt=".2f")
plt.title('10x10 Top-1% Jaccard Similarity')
plt.savefig(os.path.join(STORAGE_DIR, "FULL_plot5_jaccard_matrix.png"), dpi=300); plt.close()

# PLOT 6: Rank-vs-Rank
df['IF_pct'] = rankdata(df[f"{if_best}_score"]) / N
df['LOF_pct'] = rankdata(df[f"{lof_best}_score"]) / N
plt.figure(figsize=(9, 9))
# Sample 30k for background to prevent RAM bloat in scatter
sample_bg = df.sample(n=30000, random_state=42)
plt.scatter(sample_bg['IF_pct'], sample_bg['LOF_pct'], s=5, c='lightgray', alpha=0.3, label='Neither (Sampled BG)')
both = df[df['Group'] == 'Both']
lof_only_fraud = df[(df['Group'] == 'LOF_Only') & (df['Class'] == 1)]
plt.scatter(both['IF_pct'], both['LOF_pct'], s=20, c='blue', label='Both', edgecolors='black', linewidth=0.5)
plt.scatter(lof_only_fraud['IF_pct'], lof_only_fraud['LOF_pct'], s=50, c='red', marker='X', label='LOF-Only Fraud')
plt.axvline(x=0.99, color='black', linestyle='--', alpha=0.5)
plt.axhline(y=0.99, color='black', linestyle='--', alpha=0.5)
plt.xlabel('IF Anomaly Percentile'); plt.ylabel('LOF Anomaly Percentile')
plt.title('Rank-vs-Rank (Full Data)')
plt.legend()
plt.savefig(os.path.join(STORAGE_DIR, "FULL_plot6_rank_vs_rank.png"), dpi=300); plt.close()

# PLOT 7: PCA Agreement Map (Sampled Background)
pca = PCA(n_components=2, random_state=42)
df['PC1'], df['PC2'] = pca.fit_transform(X).T

plt.figure(figsize=(10, 8))
plt.scatter(sample_bg['PC1'], sample_bg['PC2'], s=2, c='lightgray', alpha=0.2, label='Neither (Sampled BG)')
for group, color in [('IF_Only', 'orange'), ('LOF_Only', 'green'), ('Both', 'blue')]:
    subset = df[df['Group'] == group]
    plt.scatter(subset['PC1'], subset['PC2'], s=15, c=color, label=group, edgecolor='black', linewidth=0.5)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.2%})")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.2%})")
plt.title('PCA Agreement Map (Full Data)')
plt.legend()
plt.savefig(os.path.join(STORAGE_DIR, "FULL_plot7_pca_agreement.png"), dpi=300); plt.close()

# Save Full Outputs
df.to_csv(os.path.join(STORAGE_DIR, "FULL_phase4_evaluation_dataset.csv"), index=False)
eval_df.to_csv(os.path.join(STORAGE_DIR, "FULL_phase4_metrics.csv"), index=False)
print("\nCOMPLETE PIPELINE FINISHED. All 7 plots and data saved to Drive.")

Loading FULL dataset (284,807 rows)...

--- PHASE 3: MODELING ---
Running Isolation Forest (Full Data)...
  IF_001 ... Done in 1.84s
  IF_002 ... Done in 3.11s
  IF_003 ... Done in 2.73s
  IF_004 ... Done in 1.50s
  IF_005 ... Done in 1.69s

Running LOF (Novelty Mode: Fit 50k, Predict 284k)...
  LOF_001 ... Done in 722.63s
  LOF_002 ... Done in 887.92s
  LOF_003 ... Done in 623.41s
  LOF_004 ... Done in 806.51s
  LOF_005 ... Done in 721.92s

--- PHASE 4: VALIDATION ---
 run_id           model     AP  ROC_AUC   Prec    Rec     F1  TP   FP
 IF_001 IsolationForest 0.1384   0.9451 0.1014 0.5874 0.1730 289 2560
 IF_002 IsolationForest 0.1338   0.9450 0.1000 0.5793 0.1706 285 2564
 IF_003 IsolationForest 0.2340   0.9494 0.1113 0.6443 0.1898 317 2532
 IF_004 IsolationForest 0.1487   0.9415 0.0916 0.5305 0.1562 261 2588
 IF_005 IsolationForest 0.1444   0.9485 0.1060 0.6138 0.1808 302 2547
LOF_001             LOF 0.0020   0.4901 0.0042 0.0244 0.0072  12 2837
LOF_002             LOF 0.0030   0.5

KeyError: 'PC1'

<Figure size 1000x800 with 0 Axes>

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from scipy.stats import rankdata
import warnings
warnings.filterwarnings('ignore')

STORAGE_DIR = "/content/drive/MyDrive/mplad-dvc-storage"

print("Fixing PCA assignment and generating final plots...")

# Safe PCA assignment
pca = PCA(n_components=2, random_state=42)
pc_components = pca.fit_transform(X)
df['PC1'] = pc_components[:, 0]
df['PC2'] = pc_components[:, 1]

# Safe Rank assignment
df['IF_pct'] = rankdata(df[f"{if_best}_score"]) / len(df)
df['LOF_pct'] = rankdata(df[f"{lof_best}_score"]) / len(df)

# Sample 30k for background to prevent RAM bloat in scatter
sample_bg = df.sample(n=30000, random_state=42)

# PLOT 1: PR Curves
plt.figure(figsize=(10, 6))
for run_id in run_ids:
    scores = df[f"{run_id}_score"].to_numpy()
    prec_arr, rec_arr, _ = precision_recall_curve(y_true, scores)
    if run_id == if_best: plt.plot(rec_arr, prec_arr, color='red', lw=2.5, label=f'{run_id} (Best IF)')
    elif run_id == lof_best: plt.plot(rec_arr, prec_arr, color='blue', lw=2.5, label=f'{run_id} (Best LOF)')
    elif 'IF' in run_id: plt.plot(rec_arr, prec_arr, color='red', lw=1, alpha=0.5)
    else: plt.plot(rec_arr, prec_arr, color='blue', lw=1, alpha=0.5)
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curves (Full Dataset)')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(STORAGE_DIR, "FULL_plot1_PR_curves.png"), dpi=300); plt.close()

# PLOT 2: Fraud Discovery
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
percentages = np.linspace(0.001, 0.1, 100)
for run_id, color in [(if_best, 'red'), (lof_best, 'blue')]:
    scores = df[f"{run_id}_score"].to_numpy()
    sorted_idx = np.argsort(scores)[::-1]
    y_sorted = y_true[sorted_idx]
    recalls, precs = [], []
    for p in percentages:
        k_val = int(len(y_sorted) * p)
        recalls.append(y_sorted[:k_val].sum() / y_true.sum())
        precs.append(y_sorted[:k_val].sum() / k_val)
    axes[0].plot(percentages*100, recalls, color=color, label=run_id)
    axes[1].plot(percentages*100, precs, color=color, label=run_id)
axes[0].plot(percentages*100, percentages, linestyle='--', color='gray', label='Random')
axes[1].axhline(y=y_true.sum()/N, linestyle='--', color='gray', label='Base Rate')
axes[0].set_title('Fraud Discovery (Recall vs Budget)'); axes[0].legend()
axes[1].set_title('Precision vs Budget'); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(STORAGE_DIR, "FULL_plot2_fraud_discovery.png"), dpi=300); plt.close()

# PLOT 3: Hypergeometric Null
plt.figure(figsize=(8, 5))
x = np.arange(0, 100)
pmf = hypergeom.pmf(x, N, k, k)
plt.bar(x, pmf, color='gray', alpha=0.5, label='Random Null')
plt.axvline(x=observed_overlap, color='red', linestyle='--', lw=2, label=f'Observed ({observed_overlap})')
plt.title('IF vs LOF Overlap vs Random Null (Full Data)')
plt.legend()
plt.savefig(os.path.join(STORAGE_DIR, "FULL_plot3_hypergeom.png"), dpi=300); plt.close()

# PLOT 4 & 5: 10x10 Heatmaps
plt.figure(figsize=(10, 8))
sns.heatmap(spearman_matrix.astype(float), annot=True, cmap='coolwarm', vmin=0, vmax=1, fmt=".2f")
plt.title('10x10 Spearman Rank Correlation')
plt.savefig(os.path.join(STORAGE_DIR, "FULL_plot4_spearman_matrix.png"), dpi=300); plt.close()

plt.figure(figsize=(10, 8))
sns.heatmap(jaccard_matrix.astype(float), annot=True, cmap='Blues', vmin=0, vmax=1, fmt=".2f")
plt.title('10x10 Top-1% Jaccard Similarity')
plt.savefig(os.path.join(STORAGE_DIR, "FULL_plot5_jaccard_matrix.png"), dpi=300); plt.close()

# PLOT 6: Rank-vs-Rank
plt.figure(figsize=(9, 9))
plt.scatter(sample_bg['IF_pct'], sample_bg['LOF_pct'], s=5, c='lightgray', alpha=0.3, label='Neither (Sampled BG)')
both = df[df['Group'] == 'Both']
lof_only_fraud = df[(df['Group'] == 'LOF_Only') & (df['Class'] == 1)]
plt.scatter(both['IF_pct'], both['LOF_pct'], s=20, c='blue', label='Both', edgecolors='black', linewidth=0.5)
plt.scatter(lof_only_fraud['IF_pct'], lof_only_fraud['LOF_pct'], s=50, c='red', marker='X', label='LOF-Only Fraud')
plt.axvline(x=0.99, color='black', linestyle='--', alpha=0.5)
plt.axhline(y=0.99, color='black', linestyle='--', alpha=0.5)
plt.xlabel('IF Anomaly Percentile'); plt.ylabel('LOF Anomaly Percentile')
plt.title('Rank-vs-Rank (Full Data)')
plt.legend()
plt.savefig(os.path.join(STORAGE_DIR, "FULL_plot6_rank_vs_rank.png"), dpi=300); plt.close()

# PLOT 7: PCA Agreement Map
plt.figure(figsize=(10, 8))
plt.scatter(sample_bg['PC1'], sample_bg['PC2'], s=2, c='lightgray', alpha=0.2, label='Neither (Sampled BG)')
for group, color in [('IF_Only', 'orange'), ('LOF_Only', 'green'), ('Both', 'blue')]:
    subset = df[df['Group'] == group]
    plt.scatter(subset['PC1'], subset['PC2'], s=15, c=color, label=group, edgecolor='black', linewidth=0.5)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.2%})")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.2%})")
plt.title('PCA Agreement Map (Full Data)')
plt.legend()
plt.savefig(os.path.join(STORAGE_DIR, "FULL_plot7_pca_agreement.png"), dpi=300); plt.close()

# CRITICAL: SAVE THE FINAL DATA
df.to_csv(os.path.join(STORAGE_DIR, "FULL_phase4_evaluation_dataset.csv"), index=False)
eval_df.to_csv(os.path.join(STORAGE_DIR, "FULL_phase4_metrics.csv"), index=False)
print("\nSUCCESS! All 7 plots generated and datasets saved to Drive.")

Fixing PCA assignment and generating final plots...

SUCCESS! All 7 plots generated and datasets saved to Drive.


In [ ]:
# ============================================================
# COMPARATIVE ANALYSIS: 30k Sample vs 284k Full Dataset
# ============================================================
import pandas as pd
import os

STORAGE_DIR = "/content/drive/MyDrive/mplad-dvc-storage"

# 1. Load both metric files
full_metrics = pd.read_csv(os.path.join(STORAGE_DIR, "FULL_phase4_metrics.csv"))
sample_metrics = pd.read_csv(os.path.join(STORAGE_DIR, "phase4_validation_metrics_CORRECTED.csv"))

# 2. Filter for the best post-hoc performers (IF_003 and LOF_004)
best_runs = ['IF_003', 'LOF_004']

full_best = full_metrics[full_metrics['run_id'].isin(best_runs)].copy()
sample_best = sample_metrics[sample_metrics['run_id'].isin(best_runs)].copy()

# 3. Standardize column names for merging
cols_to_keep = ['run_id', 'model', 'AP', 'ROC_AUC', 'Prec', 'Rec', 'F1', 'TP', 'FP']
full_best = full_best[cols_to_keep]

# 30k file had slightly different precision/recall naming
sample_best = sample_best[['run_id', 'model', 'AP', 'ROC_AUC', 'Prec @1%', 'Rec @1%', 'F1 @1%', 'TP', 'FP']]
sample_best.columns = cols_to_keep

# 4. Merge into a single comparison table
comparison = sample_best.merge(full_best, on=['run_id', 'model'], suffixes=('_30k', '_284k'))

print("=== FUNDAMENTAL PERFORMANCE: 30k SAMPLE vs 284k FULL ===")
# Reorder columns for readability
comparison = comparison[['run_id', 'model',
                         'AP_30k', 'AP_284k',
                         'ROC_AUC_30k', 'ROC_AUC_284k',
                         'Prec_30k', 'Prec_284k',
                         'Rec_30k', 'Rec_284k',
                         'F1_30k', 'F1_284k',
                         'TP_30k', 'TP_284k']]
print(comparison.to_string(index=False))

# 5. Derived Metrics: Consensus Overlap Comparison
print("\n=== DERIVED METRICS: CONSENSUS OVERLAP (IF_003 vs LOF_004) ===")
overlap_data = {
    'Metric': [
        'Total Rows Evaluated',
        'Total Frauds Evaluated',
        '1% Threshold (k)',
        'Expected Random Overlap',
        'Observed Overlap',
        'Enrichment Factor',
        'P-Value (Hypergeometric)'
    ],
    '30k Sample (Native fit_predict)': [
        30000, 47, 300, 3.0, 107, '35.67x', '< 10^-140'
    ],
    '284k Full (LOF Novelty Mode)': [
        284807, 492, 2848, 28.48, 457, '16.05x', '< 10^-396'
    ]
}
overlap_df = pd.DataFrame(overlap_data)
print(overlap_df.to_string(index=False))

=== FUNDAMENTAL PERFORMANCE: 30k SAMPLE vs 284k FULL ===
 run_id           model  AP_30k  AP_284k  ROC_AUC_30k  ROC_AUC_284k  Prec_30k  Prec_284k  Rec_30k  Rec_284k  F1_30k  F1_284k  TP_30k  TP_284k
 IF_003 IsolationForest  0.3126   0.2340       0.9803        0.9494    0.1000     0.1113   0.6383    0.6443  0.1729   0.1898      30      317
LOF_004             LOF  0.3229   0.0562       0.9536        0.9070    0.1267     0.0586   0.8085    0.3394  0.2190   0.1000      38      167

=== DERIVED METRICS: CONSENSUS OVERLAP (IF_003 vs LOF_004) ===
                  Metric 30k Sample (Native fit_predict) 284k Full (LOF Novelty Mode)
    Total Rows Evaluated                           30000                       284807
  Total Frauds Evaluated                              47                          492
        1% Threshold (k)                             300                         2848
 Expected Random Overlap                             3.0                        28.48
        Observed Overla